# 「미지의 전북 2026」 뉴스·SNS 관심대상 분석 노트북

**목적.** 전북 14개 시군 관련 뉴스, 인스타그램, 유튜브 자료에서 *구체적인 관심대상*(상품·장소·축제·기업·정책·인물 등)을 찾습니다. 이를 매체별로 독립 집계하고 과거와 비교해 **HOT**(현재 관심 규모가 큰 대상)과 **RISING**(과거보다 관심이 뚜렷하게 늘어난 대상)을 선정합니다.

**실행 방법.** 위에서부터 셀을 하나씩 실행하세요(`Shift+Enter`). 각 단계는 결과표를 화면에 보여주고, 마지막 단계에서 `output/` 폴더에 Excel과 CSV(UTF-8 BOM)로 저장합니다.

| 단계 | 내용 | 의뢰서 대응 |
|---|---|---|
| 0 | 환경 설정·분석 파라미터 | 10장(재현 자료) |
| 1 | 원자료 불러오기 | 2장 |
| 2 | 공통 형식으로 표준화 | 2장 |
| 3 | 데이터 현황·품질 점검 | 6.1, 10장 ① |
| 4 | 전처리(띄어쓰기 복원, 정제, 중복, 관련성) | 4.1, 4.3 |
| 5 | 관심대상 후보 추출(형태소 분석, 복합명사, 해시태그, 타임라인) | 4.2 |
| 6 | 표현 통합·필터링 → 콘텐츠×관심대상 표 | 4.1~4.3 |
| 7 | 빈도·비중 집계(매체·연도·월·시군) | 6장 |
| 8 | 6개 영역 분류와 검토표, 영역별 다빈도 키워드(8.1) | 5장 |
| 9 | HOT·RISING 선정과 변화 유형, 영역별 HOT·RISING(9.2) | 7장 |
| 10 | SNS 반응 분석 | 8장 |
| 11 | 콘텐츠 생산 주체 보조 분류 | 3.2 |
| 12 | 매체 간 공통·차별 관심대상, 맥락 비교 | 3.3, 9장 |
| 13 | 대표 사례 후보, 최종 요약표 | 10.1, 10.2 |
| 14 | 결과 저장 | 10.3 |

**폴더 구조**
```
keyword_analysis/
├── 미지의전북_키워드분석.ipynb   ← 이 노트북
├── 뉴스기사_최종유지_271159건.csv  ← 원자료(수정하지 않음)
├── 전북SNS_수집결과_20230101_20261031.xlsx  ← 원자료(수정하지 않음, 2023.1.1~10.30 보완 수집분 포함)
├── dict/     ← 불용어·보조어·지역명·표현통합·영역분류 규칙 등 (검토자가 직접 수정 가능)
├── cache/    ← 중간 결과 캐시(삭제하면 다시 계산)
└── output/   ← 최종 결과물
```

**추가 분석 규칙(2026-11-04 의뢰자 요청)**
1. 정당명(더불어민주당 등)은 분석에서 완전히 제외하고 제외 건수만 기록합니다.
2. 6개 영역별 다빈도 키워드를 매체별로 뽑습니다(복합명사 중심).
3. 과거(2023~2025) 대비 2026년 HOT·RISING을 영역별·매체별로 뽑습니다.
4. 인스타·유튜브는 '전주여행·전주맛집·전주카페' 같은 일반어를 빼고 상호명·메뉴 등 구체적인 키워드를 중심으로 봅니다.
5. '새만금'은 지역명이므로 단독으로는 빼고 새만금국제공항·새만금이차전지처럼 복합어로 추출합니다.

> **해석 원칙.** 뉴스 보도량은 언론의 주목·의제화 신호이고, SNS 언급·반응은 *수집된 플랫폼 안*에서 일어난 생산·소비 신호입니다. 이 결과를 전북 주민 전체의 관심, 실제 방문·구매량, 긍정적 인식으로 일반화하지 않습니다.

## 0. 환경 설정

In [ ]:
# ── 표준 라이브러리 ──────────────────────────────────────────────
import os, re, json, sys, time, platform, unicodedata, warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

# ── 외부 라이브러리 ──────────────────────────────────────────────
import numpy as np
import pandas as pd
from kiwipiepy import Kiwi            # 한국어 형태소 분석기
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# 그래프 한글 글꼴 (macOS: AppleGothic, Windows: Malgun Gothic, Linux: NanumGothic)
for _font in ["AppleGothic", "Malgun Gothic", "NanumGothic"]:
    if any(_font in f.name for f in matplotlib.font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = _font
        break
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# ── 경로 ─────────────────────────────────────────────────────────
BASE_DIR  = Path.cwd()                      # 노트북이 있는 폴더(keyword_analysis)에서 실행
RAW_NEWS  = BASE_DIR / "뉴스기사_최종유지_271159건.csv"
RAW_SNS   = BASE_DIR / "전북SNS_수집결과_20230101_20261031.xlsx"   # 2023.1.1~10.30 보완 수집분이 더해진 파일
DICT_DIR  = BASE_DIR / "dict"
CACHE_DIR = BASE_DIR / "cache"
OUT_DIR   = BASE_DIR / "output"
for _d in (CACHE_DIR, OUT_DIR):
    _d.mkdir(exist_ok=True)

# ── 분석 파라미터 ────────────────────────────────────────────────
# 모든 기준값을 이곳에 모았습니다. 값을 바꾸면 이후 단계에 그대로 반영됩니다.
# '잠정'으로 표시한 값은 자료 분포를 확인한 뒤 의뢰자와 협의해 확정할 값입니다.
CONFIG = {
    # 기간
    "COLLECTION_DATE": "2026-10-31",                 # SNS 수집일(누적 반응 경과일 계산 기준)
    "COMMON_START": "2023-01-01",                    # 세 매체 공통 관측기간 시작(SNS 보완 수집으로 2023-01-01부터)
    "COMMON_END":   "2026-10-26",                    # 세 매체 공통 관측기간 종료(뉴스 종료일)
    "CUR_PERIOD":  ("2026-01-01", "2026-10-26"),     # HOT·RISING: 현재 구간(2026)
    "PAST_YEARS":  [2023, 2024, 2025],               # RISING: 과거 구간 = 이 연도들의 같은 기간(1/1~10/26)을 합친 것
    "PREV_PERIOD": ("2025-01-01", "2025-10-26"),     # 참고: 전년 동기간(비중만 함께 표시)
    "SENS_PAST_YEARS": [2024, 2025],                 # 민감도: 2023년(SNS 수집 방식이 다름)을 뺀 과거 구간
    "SENS_PERIOD_MMDD": ("01-01", "08-31"),          # 민감도(인스타만): 9~10월 수집 편중을 뺀 1~8월 구간

    # 전처리
    "NEWS_DUP_WINDOW_DAYS": 3,        # 정규화한 제목이 같고 이 일수 안에 있으면 같은 기사군(전재·중복)으로 묶음
    "NEWS_EXCLUDE_REVIEW": False,     # True면 '원문확인' 판정 기사 제외
    "SNS_RELEVANCE": "standard",      # strict: 본문/위치 단서만, standard: +다른 지역 태그, loose: 전체
    "YT_DESC_MAXLEN": 1500,           # 유튜브 설명란은 앞부분만 사용(주소·링크·협찬 문구가 뒤에 몰려 있음)
    "YT_BOILERPLATE_TAG_SHARE": 0.5,  # 영상 10~29개 채널: 영상의 50% 이상에 반복되는 태그는 채널 상투 태그로 제거
    "YT_BOILERPLATE_TAG_SHARE_BIG": 0.15,  # 영상 30개 이상 채널: 15% 이상(방송사 SEO 태그 묶음 대응, 예: KBS뉴스 전북 영상 21%에 붙은 시군·K리그 태그 묶음)
    "YT_TREND_BASIS_ONLY": True,      # 연도 비교(RISING·추세)에서 시범 '추가' 수집분 제외(연도별 수집량이 달라서)

    # 후보 추출
    "NGRAM_MIN_DF": {"뉴스": 20, "인스타": 10, "유튜브": 5},   # 어절을 잇는 복합명사의 최소 등장 콘텐츠 수
    "NGRAM_ASSOC": 0.25,              # 연결 강도: df(AB) / min(df(A), df(B)) 하한
    "MAX_UNIT_LEN": 25,               # 너무 긴 단위(문장형 해시태그 등) 제외
    "TOP_N_REVIEW": 300,              # 매체별 영역분류 검토 대상 상위 후보 수

    # HOT·RISING (잠정)
    "HOT_TOP_K": 30,
    "HOT_MIN_COUNT":  {"뉴스": 50, "인스타": 30, "유튜브": 15},
    "HOT_MIN_DIVERSITY": {"뉴스": 3, "인스타": 10, "유튜브": 5},   # 언론사 / 작성자 / 채널 수
    "HOT_MIN_ACTIVE_RATIO": 0.6,      # 현재 구간 중 언급이 있었던 달의 비율
    "RISING_TOP_K": 30,
    "RISING_MIN_COUNT": {"뉴스": 30, "인스타": 20, "유튜브": 10},
    "RISING_MIN_DELTA": {"뉴스": 15, "인스타": 10, "유튜브": 5},   # 증가 건수 하한
    "RISING_MIN_LIFT": 1.5,           # 등장 비중 증가 배율 하한
    "RISING_MIN_DIVERSITY": {"뉴스": 2, "인스타": 10, "유튜브": 3},
    "RISING_MIN_ACTIVE_MONTHS": 3,
    "SPIKE_MAX_SHARE": 0.6,           # 한 달에 이 비율 이상 몰리면 '일시 급등 의존'
    "TOP_SOURCE_MAX_SHARE": 0.5,      # 한 작성자·채널·언론사가 이 비율 이상이면 '반복 생산 의존'
    "TOP3_SOURCE_MAX_SHARE": 0.8,     # 상위 3개 출처가 이 비율 이상이면 '소수 출처 집중'

    # SNS 반응
    "MIN_VIEWS_FOR_RATE": 100,        # 좋아요율·댓글률 계산 최소 조회수(소수 조회 극단값 방지)
    "HIGH_REACTION_TOP_N": 100,       # 플랫폼×유형×지표별 고반응 콘텐츠 수
    "REPEAT_AUTHOR_MIN_POSTS": 30,    # 인스타 '반복 게시 계정' 기준 게시물 수

    # 비교
    "CROSS_TOP_K": 100,               # 매체 간 비교에 쓰는 매체별 상위 관심대상 수

    # 추가 규칙(2026-11-04 의뢰자 요청)
    "SOURCE_BOILER_MIN": 10,          # 같은 채널·작성자 콘텐츠가 이 수 이상일 때 '반복 생산' 점검
    "SOURCE_BOILER_SHARE": 0.5,       # 그 출처 콘텐츠의 50% 이상에 나오는 대상은 출처당 1건만 셈
    "SOURCE_BOILER_SHARE_BIG": 0.25,  # 출처 콘텐츠가 30건 이상이면 25% 기준
    "REGION_COMPOUND_HEADS": ["새만금"],   # 단독으로는 지역명이라 빼고, 뒤에 오는 말과 묶은 복합어만 추출
    "REGION_COMPOUND_MIN_DF": {"뉴스": 10, "인스타": 5, "유튜브": 3},
    "AREA_CLASSIFY_TOP": 1500,        # 매체별로 6개 영역을 분류할 상위 후보 수
    "AREA_TOP_N": 30,                 # 영역별 다빈도 키워드 수(매체별)
    "AREA_HOT_K": 10,                 # 영역별 HOT 수(매체별)
    "AREA_RISING_K": 10,              # 영역별 RISING 수(매체별)
}
MEDIA = ["뉴스", "인스타", "유튜브"]
SIGUN = ["전주시","군산시","익산시","정읍시","남원시","김제시","완주군","진안군",
         "무주군","장수군","임실군","순창군","고창군","부안군"]
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M")

In [ ]:
def environment_info():
    """재현 정보(파이썬·라이브러리 버전, 실행 시각, 원자료 크기)를 표로 반환합니다.

    결과는 마지막 단계에서 '재현정보' 시트로 저장됩니다.
    """
    import kiwipiepy, openpyxl
    rows = [
        ("실행시각", RUN_TS),
        ("python", sys.version.split()[0]),
        ("platform", platform.platform()),
        ("pandas", pd.__version__),
        ("numpy", np.__version__),
        ("kiwipiepy", kiwipiepy.__version__),
        ("openpyxl", openpyxl.__version__),
        ("matplotlib", matplotlib.__version__),
        ("뉴스 원자료", f"{RAW_NEWS.name} ({RAW_NEWS.stat().st_size/1e6:.1f} MB)"),
        ("SNS 원자료", f"{RAW_SNS.name} ({RAW_SNS.stat().st_size/1e6:.1f} MB)"),
    ]
    return pd.DataFrame(rows, columns=["항목", "값"])

ENV_INFO = environment_info()
ENV_INFO

## 1. 원자료 불러오기

- 뉴스: CSV 1개(271,159행)
- SNS: XLSX의 `인스타`, `유튜브`, `유튜브_타임라인` 시트. 수집 범위를 해석하는 데 필요한 `수집기준`, `인스타태그범위`, `유튜브검색기준` 시트도 함께 읽습니다.
- 의뢰서 범위에 따라 **댓글 본문 시트는 분석하지 않습니다**(댓글 수만 사용).

XLSX는 읽는 데 15초쯤 걸리므로, 처음 한 번 읽은 뒤 `cache/`에 parquet으로 저장해 다음부터 빨리 불러옵니다.

In [ ]:
def _cache_path(src: Path, tag: str) -> Path:
    """원본 파일의 수정시각을 이름에 넣은 캐시 경로를 만듭니다. 원본이 바뀌면 자동으로 새로 읽습니다."""
    return CACHE_DIR / f"{tag}_{int(src.stat().st_mtime)}.parquet"


def load_news(path: Path = RAW_NEWS) -> pd.DataFrame:
    """뉴스 CSV를 문자열 그대로 읽고 추적용 '원자료_행번호'를 붙입니다.

    뉴스 자료에는 URL이나 고유 기사ID가 없으므로 의뢰서 10.2에 따라 ID를 새로 만들지 않습니다.
    대신 CSV의 행번호(헤더=1행, 첫 기사=2행)를 보존해 언론사·날짜·제목과 함께 원자료를 추적합니다.
    """
    cp = _cache_path(path, "news_raw")
    if cp.exists():
        return pd.read_parquet(cp)
    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])
    df.insert(0, "원자료_행번호", np.arange(2, len(df) + 2))
    df.to_parquet(cp)
    return df


def load_sns(path: Path = RAW_SNS) -> dict:
    """SNS 엑셀에서 분석에 쓰는 시트만 읽어 {시트명: DataFrame} 딕셔너리로 반환합니다.

    모든 열을 문자열로 읽은 뒤(앞자리 0, ID 손상 방지) 숫자 변환은 2단계에서 따로 합니다.
    """
    sheets = ["인스타", "유튜브", "유튜브_타임라인", "수집기준", "인스타태그범위", "유튜브검색기준"]
    out, missing = {}, []
    for s in sheets:
        cp = _cache_path(path, f"sns_{s}")
        if cp.exists():
            out[s] = pd.read_parquet(cp)
        else:
            missing.append(s)
    if missing:
        raw = pd.read_excel(path, sheet_name=missing, dtype=str)
        for s, df in raw.items():
            df.columns = [str(c) for c in df.columns]
            df.to_parquet(_cache_path(path, f"sns_{s}"))
            out[s] = df
    return out


t0 = time.time()
news_raw = load_news()
sns_raw = load_sns()
print(f"불러오기 완료 {time.time()-t0:.1f}초")
print("뉴스:", news_raw.shape)
for k, v in sns_raw.items():
    print(f"{k}: {v.shape}")

In [ ]:
# 수집기준 시트: 이후 해석(수집 범위·편향)의 근거가 되므로 그대로 확인합니다.
sns_raw["수집기준"].head(12)

## 2. 공통 형식으로 표준화

세 매체를 **같은 열 구조**로 맞춥니다. 합치려는 것이 아니라 *같은 함수로 따로 분석*하려는 것입니다.

| 공통 열 | 뉴스 | 인스타 | 유튜브 |
|---|---|---|---|
| `content_id` | `N`+원자료 행번호 | 게시물ID | 영상ID |
| `date` | 검색표시날짜(실제 발행일 확인 불가 → 대리 날짜) | 작성일(KST) | 게시일(KST) |
| `title` / `body` | 기사제목 / 미리보기 | – / 본문 | 제목 / 설명 |
| `tags` | – | 해시태그 | 태그 |
| `source` (다양성 지표) | 언론사 | 작성자(해시) | 채널명 |
| `sigun_all` | 검색어목록 → 시군 | 시군+관련시군 | 시군+관련시군 |
| 반응 | – | 좋아요·댓글·조회수(릴스) | 조회수·좋아요·댓글 |

In [ ]:
SIGUN_BASE = {s[:-1]: s for s in SIGUN}           # '전주' → '전주시'


def to_num(s: pd.Series) -> pd.Series:
    """문자열 숫자를 실수로 바꿉니다. 빈 값·변환 불가 값은 0이 아니라 결측(NaN)으로 둡니다."""
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")


def split_sigun(main, related) -> list:
    """'시군'과 '관련시군'(쉼표 구분)을 합쳐 중복 없는 시군 목록으로 만듭니다. 순서는 유지합니다."""
    out = []
    for v in [main] + (str(related).split(",") if isinstance(related, str) else []):
        v = str(v).strip()
        if v and v != "nan" and v not in out:
            out.append(v)
    return out


def news_keywords_to_sigun(kw_list: list) -> list:
    """뉴스 '검색어목록'(예: ["전북","전주"])을 시군 목록으로 바꿉니다.

    시군명이 없고 광역 검색어(전북, 전북특별자치도)만 있으면 SNS 표기와 맞춰 '전북 공통'으로 둡니다.
    """
    out = [SIGUN_BASE[k] for k in kw_list if k in SIGUN_BASE]
    out = list(dict.fromkeys(out))
    return out if out else ["전북 공통"]


def standardize_news(df: pd.DataFrame) -> pd.DataFrame:
    """뉴스 원자료를 공통 형식으로 바꿉니다."""
    kws = df["검색어목록"].fillna("[]").map(json.loads)
    out = pd.DataFrame({
        "media": "뉴스",
        "content_id": "N" + df["원자료_행번호"].astype(str),
        "원자료_행번호": df["원자료_행번호"],
        "date": pd.to_datetime(df["검색표시날짜"], errors="coerce"),
        "title": df["기사제목"].fillna(""),
        "body": df["미리보기"].fillna(""),
        "tags": "",
        "timeline": "",
        "location": "",
        "source": df["언론사"],
        "search_keywords": kws,
        "sigun_all": kws.map(news_keywords_to_sigun),
        "type": "기사",
        "review_flag": df["통합재판정"],            # 유지(잠정) / 원문확인 / 제외후보
        "url": "",
    })
    return out


def standardize_instagram(df: pd.DataFrame) -> pd.DataFrame:
    """인스타 시트를 공통 형식으로 바꿉니다.

    좋아요를 숨긴 게시물('좋아요숨김'=예)은 좋아요 칸에 자리표시값(대부분 3)만 들어 있으므로 결측으로 처리합니다.
    조회수는 릴스에만 제공되며, 사진·캐러셀의 조회수는 0이 아니라 '미제공(결측)'입니다.
    """
    likes = to_num(df["좋아요"]).where(df["좋아요숨김"] != "예")
    out = pd.DataFrame({
        "media": "인스타",
        "content_id": df["게시물ID"],
        "date": pd.to_datetime(df["작성일(KST)"], errors="coerce"),
        "title": "",
        "body": df["본문"].fillna(""),
        "tags": df["해시태그"].fillna(""),
        "timeline": "",
        "location": df["위치"].fillna(""),
        "source": df["작성자(해시)"],
        "sigun_all": [split_sigun(a, b) for a, b in zip(df["시군"], df["관련시군"])],
        "type": df["미디어유형"],                    # 사진 / 캐러셀 / 릴스
        "likes": likes,
        "likes_hidden": df["좋아요숨김"] == "예",
        "comments": to_num(df["댓글수"]),
        "views": to_num(df["조회수(릴스)"]),
        "collect_tag": df["수집태그"].fillna(""),
        "url": df["링크"],
    })
    return out


def standardize_youtube(df: pd.DataFrame, timeline: pd.DataFrame) -> pd.DataFrame:
    """유튜브 시트를 공통 형식으로 바꾸고, 타임라인 텍스트를 영상ID로 붙입니다.

    타임라인 행은 별도 영상으로 세지 않습니다(의뢰서 2.2). 같은 영상의 타임라인 항목을 ' | '로 이어
    'timeline' 열 하나로 만들고, 제목에 드러나지 않는 방문지·음식 등을 보완하는 데만 씁니다.
    """
    tl = timeline.copy()
    tl["텍스트"] = tl["장소명·텍스트"].fillna("").map(clean_timeline_item)
    tl = tl[tl["텍스트"] != ""]
    tl_join = tl.groupby("영상ID")["텍스트"].apply(lambda s: " | ".join(dict.fromkeys(s)))
    out = pd.DataFrame({
        "media": "유튜브",
        "content_id": df["영상ID"],
        "date": pd.to_datetime(df["게시일(KST)"], errors="coerce"),
        "title": df["제목"].fillna(""),
        "body": df["설명"].fillna(""),
        "tags": df["태그"].fillna(""),
        "timeline": df["영상ID"].map(tl_join).fillna(""),
        "location": "",
        "source": df["채널명"].fillna("").str.strip(),
        "sigun_all": [split_sigun(a, b) for a, b in zip(df["시군"], df["관련시군"])],
        "type": df["구분"],                         # 일반 / 숏츠(180초 이하)
        "views": to_num(df["조회수"]),
        "likes": to_num(df["좋아요"]),
        "comments": to_num(df["댓글수"]),
        "collect_mode": df["수집방식"],              # 기본 / 추가(시범 수집분)
        "search_query": df["검색어"].fillna(""),
        "url": df["링크"],
    })
    return out


_TL_GENERIC = re.compile(
    r"^(인트로|intro|outro|아웃트로|엔딩|ending|오프닝|opening|시작|마무리|끝|하이라이트|예고|"
    r"day\s?\d+|\d+\s?일차|첫째\s?날|둘째\s?날|셋째\s?날|넷째\s?날|\d+부|part\s?\d+|구독|광고|"
    r"클로징|closing|outro|정리|소개|요약)$", re.I)


def clean_timeline_item(s: str) -> str:
    """타임라인 항목 하나를 정리합니다. '인트로', 'Day1'처럼 대상이 아닌 구간 이름은 빈 문자열로 만듭니다."""
    s = re.sub(r"[^\w\s가-힣&·\-]", " ", str(s))
    s = re.sub(r"\s+", " ", s).strip()
    if not s or _TL_GENERIC.match(s):
        return ""
    return s


news = standardize_news(news_raw)
ig = standardize_instagram(sns_raw["인스타"])
yt = standardize_youtube(sns_raw["유튜브"], sns_raw["유튜브_타임라인"])
for d in (news, ig, yt):
    d["year"] = d["date"].dt.year
    d["month"] = d["date"].dt.to_period("M").astype(str)
print("뉴스", news.shape, "| 인스타", ig.shape, "| 유튜브", yt.shape)
print("타임라인이 붙은 유튜브 영상:", (yt["timeline"] != "").sum())

## 3. 데이터 현황·품질 점검 (산출물 ①)

결측, 중복, 기간, 매체·연도·시군 분포를 점검하고, **수집량 변화가 실제 관심 변화인지 수집 방식 때문인지** 판단할 근거를 정리합니다.

In [ ]:
def basic_quality(df: pd.DataFrame, id_col="content_id", check_cols=()) -> pd.DataFrame:
    """매체 하나의 기본 품질표(행 수, 고유 ID 수, 중복 ID, 기간, 주요 열의 결측 수·결측률)를 만듭니다."""
    rows = [
        ("원자료 행 수", len(df), ""),
        ("고유 ID 수", df[id_col].nunique(), ""),
        ("중복 ID 행", int(df[id_col].duplicated().sum()), ""),
        ("날짜 변환 실패", int(df["date"].isna().sum()), ""),
        ("최초 일자", str(df["date"].min().date()), ""),
        ("최종 일자", str(df["date"].max().date()), ""),
    ]
    for c in check_cols:
        if c not in df:
            continue
        s = df[c]
        miss = s.isna() | (s.astype(str).str.strip() == "")
        rows.append((f"결측: {c}", int(miss.sum()), f"{miss.mean():.1%}"))
    return pd.DataFrame(rows, columns=["항목", "값", "비율"])


QC = {
    "뉴스": basic_quality(news, check_cols=["title", "body", "source"]),
    "인스타": basic_quality(ig, check_cols=["body", "tags", "location", "likes", "comments", "views"]),
    "유튜브": basic_quality(yt, check_cols=["title", "body", "tags", "likes", "comments", "views", "timeline"]),
}
qc_table = pd.concat({k: v.set_index("항목") for k, v in QC.items()}, axis=1)
qc_table

In [ ]:
def year_distribution() -> pd.DataFrame:
    """매체×연도 수록 건수(원자료 기준). 2026년은 부분 연도(뉴스 10/26, SNS 10/30까지)입니다."""
    t = pd.concat([d[["media", "year"]] for d in (news, ig, yt)])
    out = t.pivot_table(index="year", columns="media", aggfunc="size", fill_value=0)[MEDIA]
    out.index.name = "연도"
    return out


def sigun_distribution() -> pd.DataFrame:
    """매체×시군 건수. 여러 시군에 걸친 콘텐츠는 *각 시군에 1번씩* 세므로 합계가 전체 건수보다 클 수 있습니다."""
    parts = []
    for d in (news, ig, yt):
        e = d[["media", "sigun_all"]].explode("sigun_all")
        parts.append(e)
    t = pd.concat(parts)
    out = t.pivot_table(index="sigun_all", columns="media", aggfunc="size", fill_value=0)[MEDIA]
    out.index.name = "시군(복수 허용)"
    return out.sort_values("뉴스", ascending=False)


def monthly_volume() -> pd.DataFrame:
    """매체별 월간 수록 건수. 수집량이 급격히 변한 달을 찾는 데 씁니다."""
    t = pd.concat([d[["media", "month"]] for d in (news, ig, yt)])
    return t.pivot_table(index="month", columns="media", aggfunc="size", fill_value=0)[MEDIA]


YEAR_DIST, SIGUN_DIST, MONTH_VOL = year_distribution(), sigun_distribution(), monthly_volume()
display(YEAR_DIST)
display(SIGUN_DIST)

ax = MONTH_VOL.loc["2023-01":].plot(figsize=(12, 4), logy=True, title="매체별 월간 수록 건수(로그 축)")
ax.set_xlabel(""); plt.tight_layout(); plt.show()

In [ ]:
def instagram_collection_bias(tag_range: pd.DataFrame) -> pd.DataFrame:
    """'인스타태그범위' 시트에서 수집이 *쪽수 상한으로 잘린* 태그를 수집구간별로 찾습니다.

    SNS 원자료는 두 구간을 따로 수집했습니다.
    - 2023.1.1~10.30(보완 수집): 해시태그의 '인기 게시물(릴스 탭)'만 받았고 '최근 게시물'은 받지 않았습니다.
    - 2023.10.31~2026.10.30: '최근 게시물'을 쪽수 상한까지 받았습니다.
    '가장 이른 작성일'이 그 구간 시작일보다 30일 넘게 늦고 쪽수 상한으로 표시된 태그는 오래된 게시물을
    끝까지 받지 못한 것입니다. 이런 태그가 많을수록 최근 월에 게시물이 몰리므로, 건수 대신 비중으로 비교합니다.
    """
    t = tag_range.dropna(subset=["수집구간"]).copy()
    t["구간시작"] = pd.to_datetime(t["수집구간"].str.extract(r"^(\d{4})\.(\d{1,2})\.(\d{1,2})")
                                  .apply(lambda r: f"{r[0]}-{r[1]}-{r[2]}", axis=1), errors="coerce")
    t["가장 이른 작성일"] = pd.to_datetime(t["가장 이른 작성일"], errors="coerce")
    t["남은 게시물"] = to_num(t["남은 게시물"])
    t["상한으로_잘림"] = (t["최근 게시물(쪽수·받은 범위)"].fillna("") + t["인기 게시물(쪽수·받은 범위)"].fillna("")).str.contains("상한")
    t["초기기간_누락"] = t["가장 이른 작성일"] > t["구간시작"] + pd.Timedelta(days=30)
    t["판정"] = np.where(t["상한으로_잘림"] & t["초기기간_누락"], "과거 게시물 일부 미수집(최근 편중)", "")
    return t.drop(columns="구간시작").sort_values(["수집구간", "남은 게시물"], ascending=[True, False])


IG_TAG_BIAS = instagram_collection_bias(sns_raw["인스타태그범위"])
n_bias = (IG_TAG_BIAS["판정"] != "").sum()
share_bias = IG_TAG_BIAS.loc[IG_TAG_BIAS["판정"] != "", "남은 게시물"].sum() / IG_TAG_BIAS["남은 게시물"].sum()
print(f"과거 게시물이 잘린 태그: {n_bias}개 / 전체 {len(IG_TAG_BIAS)}개, 해당 태그 게시물 비중(중복 포함) {share_bias:.1%}")
display(IG_TAG_BIAS.groupby("수집구간").agg(태그수=("태그", "size"), 남은게시물=("남은 게시물", "sum"),
                                         잘린태그=("판정", lambda s: (s != "").sum())))
IG_TAG_BIAS[IG_TAG_BIAS["판정"] != ""].head(15)[["수집구간", "시군", "태그", "남은 게시물", "가장 이른 작성일", "최근 게시물(쪽수·받은 범위)"]]

In [ ]:
def youtube_collection_note(df: pd.DataFrame) -> pd.DataFrame:
    """유튜브 수집방식(기본/추가)×연도 분포.

    기본 수집은 '검색어×연도마다 관련성 순 상위 50개'라서 연도별 영상 수에 *구조적인 상한*이 있습니다.
    '추가'(시범 수집분)는 2023~2025년에만 있으므로 연도 비교에서는 기본 수집분만 씁니다(CONFIG['YT_TREND_BASIS_ONLY']).
    """
    return pd.crosstab(df["collect_mode"], df["year"], margins=True, margins_name="합계")


def news_date_note() -> pd.DataFrame:
    """뉴스 날짜 열의 성격과 한계를 기록합니다(의뢰서 2.1)."""
    return pd.DataFrame([
        ("날짜 열", "검색표시날짜"),
        ("형식", "YYYY-MM-DD 단일 형식, 변환 실패 0건" if news["date"].notna().all() else "변환 실패 있음"),
        ("실제 발행일 여부", "원자료만으로는 확인 불가 → 대리 날짜로 사용(포털 검색 결과 표시일)"),
        ("분석 텍스트", "기사제목+미리보기(본문 전체 아님)"),
        ("언론사 수", f"{news['source'].nunique()}개: " + ", ".join(news['source'].value_counts().index)),
        ("통합재판정", ", ".join(f"{k} {v:,}" for k, v in news["review_flag"].value_counts().items())),
    ], columns=["항목", "내용"])


display(youtube_collection_note(yt))
display(news_date_note())

**점검 결과 해석 메모** *(실행 후 수치를 확인해 보고서에 옮기세요)*
- 인스타 2026년 수록량(특히 9~10월)은 해시태그별 '최근 글' 쪽수 상한 때문에 최근 게시물이 많이 들어온 것입니다. **건수가 아니라 기간별 등장 비중**으로 비교하고, 9~10월을 뺀 1~8월 구간으로 민감도를 점검합니다(9단계).
- **2023.1.1~10.30 보완 수집분은 수집 방식이 다릅니다.** 인스타는 해시태그 '인기 게시물(릴스 탭)'만, 유튜브는 검색어별 *조회수 순* 상위 50개를 받았습니다(이후 구간은 최근 게시물·관련성 순). 그래서 RISING은 2023~2025년을 합친 과거 구간과 비교하되, 2023년을 뺀 2024~2025년 기준으로도 조건이 유지되는지 `민감도_유지`로 표시합니다.
- 유튜브는 검색어×연도별 상위 50개 구조여서 연도별 영상 수 자체는 관심 규모를 뜻하지 않습니다. 이 역시 비중으로 비교합니다.
- 뉴스는 6개 지역 언론사에 한정되고, 제목·미리보기만으로 분석했습니다.

## 4. 전처리

### 4.1 뉴스 검색어 띄어쓰기 복원
원자료에서는 포털 검색 결과의 검색어 강조 흔적으로 **검색어 바로 뒤에 공백이 끼어 있습니다**(예: `전주 시는`, `익산 시의회`, `전북 자치도`, `전주 의`). 그대로 두면 '전주시', '익산시의회' 같은 고유명칭이 깨지므로, 그 행의 `검색어목록`에 있는 검색어 뒤의 공백 1칸만 지웁니다.

> 원래 띄어 썼던 표현(예: `전주 한옥마을`)도 붙게 되지만(`전주한옥마을`), 표현 통합 단계에서 같은 대상으로 묶이므로 집계에는 영향이 없습니다.

In [ ]:
def restore_keyword_spacing(text: str, keywords: list) -> str:
    """검색어 뒤에 잘못 들어간 공백을 지웁니다.

    Parameters
    ----------
    text : 기사제목 또는 미리보기
    keywords : 그 기사의 검색어목록(예: ["전북", "전주"])

    긴 검색어부터 처리해 '전북특별자치도'가 '전북'보다 먼저 복원되도록 합니다.
    한글·영문·숫자나 문장부호가 뒤따를 때만 공백을 지웁니다.
    """
    if not text or not keywords:
        return text
    for kw in sorted(set(keywords), key=len, reverse=True):
        text = re.sub(re.escape(kw) + r" (?=[가-힣A-Za-z0-9,.·…!?)\]’”'\"])", kw, text)
    return text


# 예시 확인
for t, k in [("익산 시의회 손진영 의원, 익산 시 수목 관리", ["익산"]), ("전북 , 소비쿠폰 찾아가는 신청", ["전북"])]:
    print(t, "→", restore_keyword_spacing(t, k))

news["title"] = [restore_keyword_spacing(t, k) for t, k in zip(news["title"], news["search_keywords"])]
news["body"] = [restore_keyword_spacing(t, k) for t, k in zip(news["body"], news["search_keywords"])]

### 4.2 텍스트 정제

In [ ]:
_URL = re.compile(r"https?://\S+|www\.\S+")
_MASK = re.compile(r"@\*+|\*{3,}")                         # 비식별 처리된 @아이디·이메일
_EMAIL = re.compile(r"\S+@\S+\.\S+")
_HASHTAG = re.compile(r"#[^\s#]+")
_TIMESTAMP = re.compile(r"\b\d{1,2}:\d{2}(:\d{2})?\b")
_NEWS_BOILER = [
    re.compile(r"[가-힣]{2,4}\s?기자"),                    # 홍길동 기자
    re.compile(r"/\s?[가-힣]{2,4}\s?=?\s?$"),               # /고창=
    re.compile(r"사진\s?=\s?\S+"),                          # 사진=전주시
    re.compile(r"\[[^\]]{1,20}\]"),                         # [전북일보], [앵커]
    re.compile(r"\([^)]{0,15}제공\)"),                      # (전주시 제공)
    re.compile(r"\.{2,}|…"),
]


def clean_common(text: str) -> str:
    """매체 공통 정제: 유니코드 정규화(NFKC), URL·이메일·비식별 기호·시각 표기 제거, 공백 정리."""
    if not isinstance(text, str) or not text:
        return ""
    t = unicodedata.normalize("NFKC", text)
    t = _URL.sub(" ", t)
    t = _EMAIL.sub(" ", t)
    t = _MASK.sub(" ", t)
    t = _TIMESTAMP.sub(" ", t)
    t = re.sub(r"[​-‏﻿]", "", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()


_STAFF_NAME = re.compile(
    r"(취재\s?기자|기자|아나운서|앵커|리포터|촬영|편집|취재|영상취재|그래픽|pd|PD)\s*[:：]?\s*[가-힣]{2,4}(\s?[,·/]\s?[가-힣]{2,4})*"
    r"|[가-힣]{2,4}\s?(기자|아나운서|앵커|리포터|PD)"
    r"|(뉴스|news|NEWS)\s*[가-힣]{2,4}\s?입니다")


def clean_youtube_text(text: str) -> str:
    """유튜브 제목·설명 정제: 공통 정제에 더해 '정다윤 기자', '촬영: 홍길동'처럼 *제작진 이름*을 지웁니다.

    방송사 뉴스 영상 설명란에는 기자·아나운서 이름이 반복되어 관심대상 순위에 들어오기 때문입니다.
    """
    return re.sub(r"\s+", " ", _STAFF_NAME.sub(" ", clean_common(text))).strip()


def clean_news_text(text: str) -> str:
    """뉴스 정제: 공통 정제에 더해 기자명, 사진 출처, 언론사 머리표 같은 상투 문구를 지웁니다."""
    t = clean_common(text)
    for p in _NEWS_BOILER:
        t = p.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()


_BOILER_LINE = re.compile(
    r"(주소|위치\s*[:：]|영업\s*시간|운영\s*시간|라스트\s*오더|브레이크\s*타임|휴무|정기휴일|전화|문의|예약|주차|"
    r"카카오톡|카톡|이메일|e-?mail|홈페이지|페이스북|인스타그램|블로그|네이버|구독|좋아요|알림\s*설정|"
    r"ⓒ|©|copyright|all\s*rights|무단\s*전재|재배포|ai\s*학습|제보|협찬\s*문의|광고\s*문의|비즈니스\s*문의|"
    r"(취재|기자|촬영|편집|그래픽|앵커|영상)\s*[:：/]|영상\s*제공|화면\s*제공|신청\s*방법|신청\s*기간|모집\s*기간|모집\s*대상|참가비|선착순|당첨자|qr|music\s*provided|bgm)", re.I)


def drop_boiler_lines(text: str) -> str:
    """SNS 본문·유튜브 설명란에서 연락처·주소·영업시간·저작권·구독 요청·이벤트 신청 안내 같은 *안내 문구 줄*을 지웁니다.

    이런 줄은 관심대상이 아니라 게시 형식에서 반복되는 문구라서 빈도 상위를 차지합니다('홈페이지', '1층', '무단전재' 등).
    줄바꿈 단위로 판단하므로 정제(공백 합치기) *전에* 적용합니다.
    """
    if not isinstance(text, str) or not text:
        return ""
    return "\n".join(l for l in text.splitlines() if not _BOILER_LINE.search(l) and not is_stuffed_line(l))


def is_stuffed_line(line: str, min_tokens: int = 8, min_share: float = 0.6) -> bool:
    """키워드 도배 줄인지 판정합니다. 예: '익산석재.김제석재.부안석재.고창석재.…'

    구분자(. , · / | 공백)로 나눈 조각이 min_tokens개 이상이고, 조각 끝 2글자가 같은 것이 min_share 이상이면
    검색 노출용으로 지명을 나열한 줄로 보고 지웁니다.
    """
    parts = [p for p in re.split(r"[.,·/|\s]+", line) if len(p) >= 3]
    if len(parts) < min_tokens:
        return False
    top = Counter(p[-2:] for p in parts).most_common(1)[0][1]
    return top / len(parts) >= min_share


def extract_hashtags(text: str) -> list:
    """문자열에서 해시태그를 뽑아 '#'과 끝 문장부호를 떼고 반환합니다. 같은 태그는 한 번만 남깁니다."""
    if not isinstance(text, str):
        return []
    tags = [re.sub(r"[^\w가-힣]+$", "", h[1:]) for h in _HASHTAG.findall(unicodedata.normalize("NFKC", text))]
    return list(dict.fromkeys(t for t in tags if t))


def split_yt_tags(text: str) -> list:
    """유튜브 '태그' 칸(쉼표 구분)을 목록으로 만듭니다.

    방송사 채널은 태그 끝에 '아나운서, 홍길동, 김철수'처럼 진행자 이름을 붙이므로, '아나운서·앵커·기자' 태그
    *뒤에 오는* 2~4글자 한글 태그는 인명으로 보고 뺍니다. 태그 칸에 섞인 URL도 지웁니다.
    """
    if not isinstance(text, str) or not text.strip():
        return []
    out, after_staff = [], False
    for t in unicodedata.normalize("NFKC", text).split(","):
        t = _URL.sub("", t).strip()
        if not t:
            continue
        if t in ("아나운서", "앵커", "기자", "리포터"):
            after_staff = True
            continue
        if after_staff and re.fullmatch(r"[가-힣]{2,4}", t):
            continue
        after_staff = False
        out.append(t)
    return list(dict.fromkeys(out))


def collect_staff_names(texts) -> set:
    """유튜브 설명란에서 기자·제작진·영상제공자 이름을 모읍니다(태그·본문에서 이 이름을 빼는 데 씀).

    잡는 꼴: '임홍진 기자', 'MBC뉴스 조수영입니다', '영상제공: 남원시(강석현), 완주군(김회성)', '촬영: 홍길동'.
    인명처럼 보이는 3글자(성씨로 시작)이고 2번 이상 나온 이름만 남겨 '사진기자' 같은 일반어가 섞이지 않게 합니다.
    """
    pats = [re.compile(r"([가-힣]{3})\s?(기자|아나운서|앵커|리포터|PD)"),
            re.compile(r"(?:뉴스|news)\s*([가-힣]{3})\s?입니다", re.I),
            re.compile(r"(?:취재|촬영|편집|영상취재|그래픽)\s*[:：]?\s*([가-힣]{3})")]
    credit = re.compile(r"(?:영상|화면)\s*제공\s*[:：]?([^\n]*)")
    names = Counter()
    for t in texts:
        if not isinstance(t, str):
            continue
        for p in pats:
            names.update(m.group(1) for m in p.finditer(t))
        for c in credit.finditer(t):
            names.update(re.findall(r"\(([가-힣]{3})\)", c.group(1)))
    return {n for n, c in names.items() if c >= 2 and n[0] in _SURNAMES_STAFF}


_SURNAMES_STAFF = set("김이박최정강조윤장임한오서신권황안송류유홍전고문양손배백허남심노하곽성차주우구민진나지엄채원천방공현함변염여추도소석선설마길연위표명기반왕금옥육인맹제모탁국어은편용예경봉")
YT_STAFF = collect_staff_names(yt["body"])
print(f"유튜브 제작진·영상제공자 이름 {len(YT_STAFF)}개 수집(태그·대상에서 제외):", ", ".join(sorted(YT_STAFF)[:15]), "…")

# 뉴스: 제목 + 미리보기
news["text"] = (news["title"].map(clean_news_text) + " . " + news["body"].map(clean_news_text)).str.strip(" .")
news["tag_list"] = [[] for _ in range(len(news))]

# 인스타: 본문에서 해시태그를 떼어 내 따로 보관(본문 형태소 분석과 태그 분석이 겹치지 않도록)
ig["tag_list"] = [list(dict.fromkeys(extract_hashtags(a) + extract_hashtags(b))) for a, b in zip(ig["tags"], ig["body"])]
ig["text"] = (ig["body"].map(drop_boiler_lines).map(lambda s: _HASHTAG.sub(" ", s)).map(clean_common) + " . " + ig["location"].map(clean_common)).str.strip(" .")

# 유튜브: 제목 + 설명(앞부분) + 타임라인, 태그는 따로
yt["tag_list"] = [[t for t in dict.fromkeys(split_yt_tags(a) + extract_hashtags(b) + extract_hashtags(c)) if t not in YT_STAFF]
                  for a, b, c in zip(yt["tags"], yt["title"], yt["body"])]
yt["text"] = (yt["title"].map(lambda s: _HASHTAG.sub(" ", s)).map(clean_youtube_text) + " . "
              + yt["body"].map(drop_boiler_lines).str[:CONFIG["YT_DESC_MAXLEN"]].map(lambda s: _HASHTAG.sub(" ", s)).map(clean_youtube_text) + " . "
              + yt["timeline"].map(clean_common)).str.strip(" .")

print(news[["title", "text"]].head(3).to_string()[:600])

In [ ]:
def remove_channel_boilerplate_tags(df: pd.DataFrame, share: float, min_videos: int = 10,
                                    share_big: float = 0.25, big_videos: int = 30):
    """유튜브 채널이 모든 영상에 반복해서 붙이는 상투 태그(예: '전주MBC, 뉴스데스크, 속보')를 지웁니다.

    영상이 min_videos개 이상인 채널에서, 그 채널 영상의 share 비율(영상이 big_videos개 이상이면 share_big) 이상에
    붙은 태그를 '채널 상투 태그'로 봅니다. 방송사 채널은 아나운서 이름까지 포함한 SEO 태그 묶음을 일부 영상에만
    붙이는 경우가 있어 큰 채널에는 더 낮은 기준을 씁니다.
    이런 태그는 관심대상이 아니라 채널 표식이므로 빈도에서 빼고, 제거 목록을 결과로 남깁니다.
    """
    removed, bp_by_channel = [], {}
    for ch, g in df.groupby("source"):
        if len(g) < min_videos:
            continue
        cnt = Counter(t for tl in g["tag_list"] for t in set(tl))
        thr = share_big if len(g) >= big_videos else share
        bp = {t for t, c in cnt.items() if c / len(g) >= thr}
        if bp:
            bp_by_channel[ch] = bp
            removed += [(ch, len(g), t, cnt[t]) for t in sorted(bp)]
    new_lists = [[t for t in tl if t not in bp_by_channel.get(ch, ())] for ch, tl in zip(df["source"], df["tag_list"])]
    return new_lists, pd.DataFrame(removed, columns=["채널명", "채널 영상 수", "제거 태그", "붙은 영상 수"])


yt["tag_list"], YT_BOILER = remove_channel_boilerplate_tags(yt, CONFIG["YT_BOILERPLATE_TAG_SHARE"],
                                                             share_big=CONFIG["YT_BOILERPLATE_TAG_SHARE_BIG"])
print(f"채널 상투 태그 {len(YT_BOILER)}건 제거 ({YT_BOILER['채널명'].nunique()}개 채널)")
YT_BOILER.head(10)

### 4.3 뉴스 중복·전재 처리
- **완전 중복:** 같은 언론사, 같은 날짜, 같은 제목(점검 결과 0건)
- **기사군:** 기호·공백을 지운 제목이 같고 `NEWS_DUP_WINDOW_DAYS`일 안에 실린 기사는 같은 사건을 다룬 **동일 기사·전재**로 보고 한 기사군으로 묶습니다. 뉴스 빈도는 **기사군 수**로 세고, 언론사 다양성은 기사군에 속한 모든 언론사로 계산합니다.
- 제목이 조금씩 다른 유사 기사는 묶지 않았습니다(한계로 기록).

In [ ]:
def mark_news_duplicates(df: pd.DataFrame, window_days: int) -> pd.DataFrame:
    """정규화한 제목과 날짜 간격으로 뉴스 기사군(dup_group)을 붙입니다.

    처리 순서: (1) 제목에서 한글·영문·숫자만 남겨 norm_title을 만들고, (2) norm_title별로 날짜순 정렬한 뒤,
    (3) 앞 기사와의 간격이 window_days를 넘으면 새 기사군을 시작합니다.
    제목이 8자 미만이면('[포토]', '인사' 등) 잘못 묶일 수 있어 기사마다 따로 기사군을 줍니다.

    Returns
    -------
    dup_group, dup_size(기사군 크기), dup_press_n(기사군 언론사 수) 열이 추가된 DataFrame
    """
    d = df.copy()
    d["norm_title"] = d["title"].str.replace(r"[^0-9A-Za-z가-힣]", "", regex=True)
    d = d.sort_values(["norm_title", "date"])
    new_grp = (d["norm_title"] != d["norm_title"].shift()) | (d["date"].diff().dt.days > window_days) \
              | (d["norm_title"].str.len() < 8)
    d["dup_group"] = "G" + new_grp.cumsum().astype(str)
    g = d.groupby("dup_group")
    d["dup_size"] = g["content_id"].transform("size")
    d["dup_press_n"] = g["source"].transform("nunique")
    return d.sort_index()


news = mark_news_duplicates(news, CONFIG["NEWS_DUP_WINDOW_DAYS"])
NEWS_DUP_SUMMARY = pd.DataFrame([
    ("원자료 기사 수", len(news)),
    ("완전 중복(언론사·날짜·제목 동일)", int(news.duplicated(["source", "date", "title"]).sum())),
    ("기사군 수(빈도 산정 기준)", news["dup_group"].nunique()),
    ("2건 이상 묶인 기사군", int((news.groupby("dup_group").size() > 1).sum())),
    ("묶여서 줄어든 기사 수", len(news) - news["dup_group"].nunique()),
    ("여러 언론사가 같은 제목으로 실은 기사군", int((news.groupby("dup_group")["source"].nunique() > 1).sum())),
], columns=["항목", "건수"])
NEWS_DUP_SUMMARY

### 4.4 SNS 지역 관련성 점검
수집 태그·검색어가 붙었다는 사실만으로 전북 관련 콘텐츠로 확정하지 않습니다(의뢰서 4.3). 콘텐츠마다 지역 단서의 수준을 표시합니다.

| 수준 | 기준 |
|---|---|
| A | 본문·위치(유튜브는 제목·설명·타임라인)에 시군명, 전북, 대표 지명 등 지역 단서가 있음 |
| B | 본문에는 없지만 수집 태그 *이외의* 해시태그·태그에 지역 단서가 있음 |
| C | 수집 태그 말고는 지역 단서가 없음 |

`CONFIG["SNS_RELEVANCE"]`: `strict`=A, `standard`=A+B(기본값), `loose`=A+B+C

In [ ]:
def build_region_cue_pattern() -> re.Pattern:
    """지역 단서 정규식: 시군명, 광역명, 수집 태그로 쓰인 대표 지명·축제·특산물, 영문 지명."""
    words = list(SIGUN_BASE.keys()) + ["전북", "전라북도", "새만금", "jeonbuk", "jeollabuk"]
    words += ["고군산", "선유도", "미륵사", "왕궁리", "내장산", "광한루", "지평선", "삼례", "마이산", "덕유산",
              "반딧불", "임실치즈", "순창고추장", "강천산", "선운사", "청보리", "변산", "채석강", "내소사", "격포",
              "곰소", "한옥마을", "원광대", "군산대", "전북대", "장안산", "모악산", "대둔산", "운일암", "구천동"]
    words += ["jeonju", "gunsan", "iksan", "jeongeup", "namwon", "gimje", "wanju", "jinan", "muju",
              "jangsu", "imsil", "sunchang", "gochang", "buan"]
    return re.compile("|".join(sorted(map(re.escape, words), key=len, reverse=True)), re.I)


REGION_CUE = build_region_cue_pattern()


def relevance_level(text: str, tags: list, collect_tags: str = "") -> str:
    """콘텐츠 하나의 지역 단서 수준(A/B/C)을 판정합니다. 기준은 위 표와 같습니다."""
    if REGION_CUE.search(text or ""):
        return "A"
    ct = {t.strip().lstrip("#") for t in re.split(r"[,\s]+", collect_tags or "") if t.strip()}
    if any(REGION_CUE.search(t) for t in tags if t not in ct):
        return "B"
    return "C"


ig["relevance"] = [relevance_level(t, tl, ct) for t, tl, ct in zip(ig["text"], ig["tag_list"], ig["collect_tag"])]
yt["relevance"] = [relevance_level(t, tl) for t, tl in zip(yt["text"], yt["tag_list"])]
news["relevance"] = "A"   # 뉴스는 원자료 단계에서 관련성 판정(최종판정=유지)을 마쳤으므로 그대로 사용

REL_TABLE = pd.concat({
    "인스타": ig["relevance"].value_counts(), "유튜브": yt["relevance"].value_counts()
}, axis=1).fillna(0).astype(int).sort_index()
display(REL_TABLE)
print("C 수준 인스타 예시:")
for s in ig.loc[ig["relevance"] == "C", "body"].head(3):
    print(" -", s[:120].replace("\n", " "))

In [ ]:
def apply_validity(df: pd.DataFrame) -> pd.Series:
    """CONFIG 기준에 따라 분석에 포함할 콘텐츠(유효 콘텐츠) 여부를 True/False로 돌려줍니다.

    - 공통: 날짜가 있고 공통 관측기간 이전 자료도 포함(뉴스 장기 분석용). 기간 제한은 집계 단계에서 적용
    - SNS: 지역 관련성 수준 필터
    - 뉴스: NEWS_EXCLUDE_REVIEW=True면 '원문확인' 판정 제외
    """
    ok = df["date"].notna()
    mode = CONFIG["SNS_RELEVANCE"]
    if df["media"].iat[0] in ("인스타", "유튜브"):
        allowed = {"strict": {"A"}, "standard": {"A", "B"}, "loose": {"A", "B", "C"}}[mode]
        ok &= df["relevance"].isin(allowed)
    if df["media"].iat[0] == "뉴스" and CONFIG["NEWS_EXCLUDE_REVIEW"]:
        ok &= df["review_flag"] != "원문확인"
    return ok


def mark_sns_text_duplicates(df: pd.DataFrame, min_len: int = 30) -> pd.Series:
    """SNS에서 본문이 사실상 같은 게시물(같은 홍보문·이벤트 공지를 여러 번 올린 경우)을 하나의 묶음으로 표시합니다.

    해시태그·기호·공백을 지운 본문 앞 200자가 같으면 같은 묶음입니다. 본문이 min_len자 미만이면
    잘못 묶일 수 있어 게시물마다 따로 둡니다. 반환값은 빈도 산정 단위(count_key)로 씁니다.
    """
    norm = df["body"].map(lambda s: _HASHTAG.sub("", s or "")).str.replace(r"[^0-9A-Za-z가-힣]", "", regex=True).str[:200]
    key = np.where(norm.str.len() >= min_len, "T" + norm, "C" + df["content_id"])
    return pd.Series(pd.factorize(key)[0], index=df.index).map(lambda i: f"{df['media'].iat[0][0]}{i}")


for d in (news, ig, yt):
    d["valid"] = apply_validity(d)
    # 빈도 산정 단위: 뉴스=기사군, 인스타=동일 본문 묶음, 유튜브=영상
    if d["media"].iat[0] == "뉴스":
        d["count_key"] = d["dup_group"]
    elif d["media"].iat[0] == "인스타":
        d["count_key"] = mark_sns_text_duplicates(d)
    else:
        d["count_key"] = d["content_id"]
print("인스타 동일 본문 반복으로 줄어든 게시물:", len(ig) - ig["count_key"].nunique())

VALID_SUMMARY = pd.DataFrame({
    m: {"원자료": len(d), "유효 콘텐츠": int(d["valid"].sum()), "빈도 산정 단위 수": d.loc[d["valid"], "count_key"].nunique()}
    for m, d in zip(MEDIA, (news, ig, yt))
}).T
VALID_SUMMARY

## 5. 관심대상 후보 추출

**방법.** Kiwi 형태소 분석기로 명사를 찾은 뒤 다음 세 종류의 의미 단위를 만듭니다.

1. **어절 내 명사 연쇄**: 띄어 쓰지 않은 명사 묶음을 한 단위로 봅니다. 예: `군산시간여행축제`, `전주비빔밥`, `임실치즈테마파크`
2. **어절을 잇는 복합명사**: 조사 없이 이어진 명사 어절(예: `새만금 국제공항`)은 *자주 함께 나오고 결합이 강할 때만* 한 단위로 잇습니다(`NGRAM_MIN_DF`, `NGRAM_ASSOC`). 명사를 기계적으로 이어 붙이는 것(예: `예산 확보 추진`)을 막기 위해서입니다.
3. **해시태그·유튜브 태그**: 작성자가 직접 붙인 단위를 그대로 씁니다.

유튜브 타임라인은 2.에서 본문 텍스트에 포함했으므로 같은 방식으로 추출됩니다.

> 이 단계는 시간이 가장 오래 걸립니다(전체 약 수 분). 결과는 `cache/`에 저장되어 다시 실행하면 바로 불러옵니다.

In [ ]:
def load_word_list(name: str) -> set:
    """dict/ 폴더의 한 줄 한 단어 텍스트 사전을 읽습니다. '#'으로 시작하는 줄은 설명입니다."""
    lines = (DICT_DIR / name).read_text(encoding="utf-8").splitlines()
    return {normalize_unit(l) for l in lines if l.strip() and not l.startswith("#")}


def normalize_unit(s: str) -> str:
    """후보 단위의 표기를 통일합니다.

    NFKC 정규화 → 영문 소문자화 → 한글·영문·숫자 이외 문자 제거(띄어쓰기 포함) →
    앞에 붙은 연도·회차 표기 제거(예: '2026군산시간여행축제' → '군산시간여행축제', '제27회' 제거)
    """
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = re.sub(r"[^0-9a-z가-힣]", "", s)
    s = re.sub(r"^((19|20)\d{2}(년)?|제?\d{1,3}(회|차|기|번째))+", "", s)
    s = re.sub(r"(?<=[가-힣])\d+$", "", s)                      # '전주시의회5' → '전주시의회'
    s = re.sub(r"(?<=[가-힣])[a-z]{4,}$", "", s)                 # '한옥마을jeonjuhanok' → '한옥마을'(영문 4자 이상 꼬리)
    return s


def make_kiwi() -> Kiwi:
    """Kiwi 분석기를 만들고 dict/사용자사전.csv의 고유명사를 등록합니다(지명·축제명이 쪼개지지 않도록)."""
    kw = Kiwi(num_workers=-1)      # -1 = 사용 가능한 모든 CPU 코어(0이면 단일 스레드)
    ud = pd.read_csv(DICT_DIR / "사용자사전.csv")
    for w, tag in zip(ud["단어"], ud["품사"]):
        kw.add_user_word(str(w), tag, 0.0)
    return kw


kiwi = make_kiwi()
NOUN_TAGS = {"NNG", "NNP", "SL", "SN", "SH"}
XSN_KEEP_EXCLUDE = {"들", "적", "님", "씨", "등", "께"}
print([ (t.form, t.tag) for t in kiwi.tokenize("군산시간여행축제와 새만금 국제공항, 전주비빔밥 맛집") ])

In [ ]:
def _is_noun(tok) -> bool:
    """명사 연쇄에 포함할 토큰인지 판단합니다. 일반·고유명사, 외국어, 숫자, 한자와 일부 명사 파생 접미사를 포함합니다."""
    if tok.tag in NOUN_TAGS:
        return True
    if tok.tag == "XSN" and tok.form not in XSN_KEEP_EXCLUDE:
        return True
    if tok.tag == "XPN":
        return True
    return False


def extract_runs(tokens, text: str):
    """형태소 분석 결과에서 '어절 내 명사 연쇄'와 '어절을 잇는 인접 연쇄 쌍·3연쇄'를 뽑습니다.

    Returns
    -------
    runs : 어절 내 명사 연쇄의 원문 표기 목록(예: ['군산시간여행축제', '새만금', '국제공항'])
    ngrams : 공백 하나로만 떨어져 이어진 연쇄들의 튜플 목록(예: [('새만금','국제공항')])
             앞 연쇄 뒤에 조사·어미 등이 붙으면 끊어진 것으로 보고 잇지 않습니다.
    """
    runs, cur = [], []
    for t in tokens:
        if _is_noun(t):
            if cur and t.start == cur[-1].end:           # 공백 없이 붙어 있음 → 같은 어절의 명사 연쇄
                cur.append(t)
            else:
                if cur:
                    runs.append(cur)
                cur = [t]
        else:
            if cur:
                runs.append(cur)
                cur = []
            runs.append(None)                             # 명사가 아닌 토큰 = 연결 끊김 표시
    if cur:
        runs.append(cur)

    surfaces, ngrams, chain = [], [], []
    prev_end = None
    for r in runs:
        if r is None:
            chain, prev_end = [], None
            continue
        s = text[r[0].start:r[-1].end]
        surfaces.append(s)
        gap = text[prev_end:r[0].start] if prev_end is not None else None
        if gap is not None and gap.strip() == "" and len(gap) >= 1:
            chain.append(s)
        else:
            chain = [s]
        if len(chain) >= 2:
            ngrams.append(tuple(chain[-2:]))
        if len(chain) >= 3:
            ngrams.append(tuple(chain[-3:]))
        prev_end = r[-1].end
    return surfaces, ngrams


def tokenize_media(df: pd.DataFrame, tag: str) -> pd.DataFrame:
    """매체 하나의 모든 유효 콘텐츠를 형태소 분석해 콘텐츠별 단위 목록을 만듭니다(캐시 사용).

    Returns
    -------
    DataFrame[content_id, runs(list), ngrams(list of tuple)]
    """
    import hashlib
    sig = hashlib.md5(("|".join(df["content_id"]) + "\x1e".join(df["text"])).encode()).hexdigest()[:10]   # 텍스트 전체 기준
    cp = CACHE_DIR / f"units_{tag}_{len(df)}_{sig}.pkl"     # 콘텐츠·텍스트가 바뀌면 캐시 이름도 바뀜
    if cp.exists():
        return pd.read_pickle(cp)
    texts = df["text"].fillna("").tolist()
    t0 = time.time()
    res = []
    for text, toks in zip(texts, kiwi.tokenize(texts)):
        res.append(extract_runs(toks, text))
    out = pd.DataFrame({"content_id": df["content_id"].values,
                        "runs": [r[0] for r in res], "ngrams": [r[1] for r in res]})
    out.to_pickle(cp)
    print(f"  {tag}: {len(df):,}건 형태소 분석 {time.time()-t0:.0f}초")
    return out


# 예시
_t = "새만금 국제공항 건설 예산 확보를 추진한다. 군산시간여행축제 개막"
print(extract_runs(kiwi.tokenize(_t), _t))

In [ ]:
UNITS = {}
for m, d in zip(MEDIA, (news, ig, yt)):
    UNITS[m] = tokenize_media(d[d["valid"]], m)
    print(m, "완료:", len(UNITS[m]))

## 6. 표현 통합·필터링 → 콘텐츠×관심대상 표

각 단위를 정규화한 뒤 다음 순서로 판정합니다.

| 판정 | 기준 | 처리 |
|---|---|---|
| 정당명 | `dict/제외_정당명.txt`의 정당명(더불어민주당, 민주당, 국민의힘 …)이 들어간 단위 | **분석에서 완전히 제외**(순위·영역·HOT·RISING 모두), 제외 건수만 따로 기록 |
| 새만금 복합어 | `새만금` 단독은 지역명으로 빼고, 뒤에 오는 말과 묶인 복합어(새만금국제공항, 새만금이차전지 …)만 후보로 남김 | 후보 |
| 지역명 | `dict/지역명.txt`에 정확히 일치(전주, 전주시, 전북특별자치도 …), 하위 행정구역(완산구, 전주시완산구, ○○읍·면·동) | 순위에서 제외, 시군 정보로만 보존 |
| 불용어 | `dict/불용어.txt`, 숫자만, 1글자 | 제외 |
| 보조어 | `dict/보조어.txt`(관광·경제·지원·축제 …), *시군명+일반어*(전주맛집, 군산여행), SNS 상투 태그, 화이트리스트에 없는 2글자 단일 일반명사·영문 일반어, **14개 시군에 고르게 퍼진 일반명사(범용어 자동 판정)** | 주요 순위에서 제외, 보조어 집계표로 따로 제공 |
| 후보 | 그 밖의 모든 단위 | **관심대상 후보** |

지역명이 들어간 고유명칭(`전주비빔밥`, `군산시간여행축제`)은 지역명이 일부일 뿐이므로 지우지 않습니다(의뢰서 4.1).

**SNS(인스타·유튜브) 추가 규칙.** `전주여행`, `전주맛집`, `군산카페`, `감성카페`, `빵지순례`처럼 *어디에나 붙는 여행·맛집·카페 일반어*는 보조어로 뺍니다(`dict/SNS일반어.txt`와 일반어 결합 패턴). `전주맛집○○`처럼 *시군명+일반어 뒤에 상호·메뉴가 붙은 태그*는 앞부분을 떼고 뒤의 상호·메뉴만 후보로 남깁니다.

In [ ]:
REGION_WORDS = load_word_list("지역명.txt")
PARTY_WORDS = load_word_list("제외_정당명.txt")            # 규칙 1: 정당명은 분석에서 제외
SNS_WORDS = load_word_list("SNS일반어.txt")                # 규칙 4: SNS 여행·맛집·카페 일반어
_PARTY = re.compile("|".join(sorted(map(re.escape, PARTY_WORDS), key=len, reverse=True)))
_HEADS = sorted({normalize_unit(h) for h in CONFIG["REGION_COMPOUND_HEADS"]}, key=len, reverse=True)
_HEAD_RE = re.compile(r"^(전북특별자치도|전북자치도|전라북도|전북도|전북)?(" + "|".join(map(re.escape, _HEADS)) + ")")
_HEAD_GENERIC_TAIL = {"지역", "일대", "권역", "인근", "주변", "일원", "관련", "내", "쪽", "지구", "현장", "문제", "사업", "개발", "개발사업"}
STOP_WORDS = load_word_list("불용어.txt")
AUX_WORDS = load_word_list("보조어.txt")
WHITELIST = load_word_list("화이트리스트.txt")
USER_WORDS = {normalize_unit(w) for w in pd.read_csv(DICT_DIR / "사용자사전.csv")["단어"]}
SYN = pd.read_csv(DICT_DIR / "표현통합사전.csv")
SYN_MAP = {normalize_unit(a): str(b) for a, b in zip(SYN["원표현"], SYN["표준명칭"])}

_REGION_PREFIX = re.compile(r"^(" + "|".join(sorted(
    [r for r in REGION_WORDS if re.fullmatch(r"[가-힣]+", r)], key=len, reverse=True)) + r")")
_SNS_GENERIC = re.compile(
    r"(스타그램|그램|stagram|투어|추천|맞팔|선팔|팔로우|좋아요|소통|일상|데일리|daily|여행|여행지|여행코스|"
    r"가볼만한곳|가볼만한|가볼곳|핫플|핫플레이스|데이트|나들이|카페|맛집|먹방|브이로그|vlog|travel|trip|"
    r"food|cafe|tour|korea|photo|instagood|추억|기록|주말|휴가|당일치기)$")
# 규칙 4: '수식어* + 업종·여행 일반어' 꼴(감성카페, 대형카페, 오션뷰카페, 현지인맛집, 빵지순례, 카페투어 …)은 SNS에서 보조어
_SNS_MOD = (r"(감성|대형|신상|분위기|오션뷰|리버뷰|마운틴뷰|뷰|루프탑|한옥|애견동반|애견|반려견동반|반려견|댕댕이|아이랑|아이와|가족|커플|"
            r"혼밥|혼술|로컬|현지인|노포|숨은|인생|동네|가성비|무한리필|수제|정원|산속|시골|바다|바닷가|호수|숲속|키즈|노키즈|"
            r"가볼만한|당일|주말|데이트|드라이브|단풍|벚꽃|가을|봄|여름|겨울|국내|전국|전라도|호남|이색|예쁜|유명|핫한|조용한|"
            r"베이커리|디저트|브런치|빵|커피|술|밥|고기|회|한식|양식|중식|일식|분식|레트로|빈티지|모던|넓은|큰|작은|"
            r"신규|오픈|그랜드오픈|오픈런|웨이팅|줄서는|존맛|찐|jmt|대박|최고|추천|소개|후기|리뷰|방문)")
_SNS_HEAD = (r"(카페|맛집|식당|빵집|베이커리|술집|밥집|횟집|고깃집|브런치|디저트|여행|여행지|코스|투어|명소|핫플|핫플레이스|"
             r"데이트|데이트코스|나들이|산책|산책로|캠핑|글램핑|차박|숙소|펜션|숙박|호텔|드라이브|스냅|포토존|포토스팟|스팟|"
             r"야경|단풍|단풍명소|벚꽃|꽃구경|꽃놀이|빵지순례|카페투어|맛집투어|맛집탐방|카페탐방|먹스타그램|카페스타그램|"
             r"빵스타그램|여행스타그램|럽스타그램|일상스타그램|소품샵|편집샵|공방|원데이클래스|플리마켓|팝업|팝업스토어)")
_SNS_GENERIC_PLACE = re.compile(r"^" + _SNS_MOD + r"*" + _SNS_HEAD + r"(" + _SNS_MOD + r"|" + _SNS_HEAD + r"|스타그램|그램|탐방|순례|추천|투어)*$")
_REGION_GENERIC_PREFIX = None    # 시군명+일반어 접두(전주맛집○○ → ○○), 아래에서 지역명 목록으로 만듦
_TITLE_NAME = re.compile(r"(news|은행장|이사장|본부장|원장|사장|회장|청장|시장|군수|의장|위원장|교육감|총장|대표|교수)([가-힣]{3})")
_ADMIN_UNIT = re.compile(r"((전북|전라북도|전북특별자치도)?[가-힣]{2}(시|군))+[가-힣]{1,3}(읍|면)([가-힣]{1,3}리)?|[가-힣]{2}읍([가-힣]{1,3}리)?")   # '짜장면' 같은 음식명은 제외
_PARTICLES = {"이", "가", "은", "는", "을", "를", "의", "에", "서", "와", "과", "도", "로", "에서", "으로"}
_SUBREGION = re.compile(r"(완산구|덕진구|[가-힣]{1,3}(읍|면)|(전주시|군산시|익산시|정읍시|남원시|김제시)?[가-힣]{1,3}동|(시|군)(완산구|덕진구)?)")
_TITLE_TAIL = re.compile(r"(시장|군수|도지사|지사|의원|국회의원|교육감|위원장|의장|후보|예비후보|대표|총장|청장|본부장|이사장|회장)$")
_VERBAL_TAIL = {"추진", "개최", "실시", "진행", "마련", "확대", "강화", "선정", "참여", "운영", "지원", "협약",
                "체결", "발표", "확보", "건의", "촉구", "논의", "점검", "모집", "시행", "완료", "착수", "준공", "개막", "폐막"}

print(f"정당명 {len(PARTY_WORDS)} | SNS일반어 {len(SNS_WORDS)} | 지역명 {len(REGION_WORDS)} | 불용어 {len(STOP_WORDS)} | 보조어 {len(AUX_WORDS)} | 화이트리스트 {len(WHITELIST)} | 통합사전 {len(SYN_MAP)}")

In [ ]:
from functools import lru_cache
SYN_STD = {normalize_unit(v) for v in SYN_MAP.values()}
_UNIT_INFO_CACHE = {}


def unit_token_info(u: str):
    """정규화된 단위를 다시 형태소 분석해 (명사 토큰 수, 고유명사 포함 여부)를 돌려줍니다(결과 캐시).

    '2글자 단일 일반명사는 보조어'라는 규칙을 적용하려고 쓰며, 고유명사(NNP)로 분석되면 후보로 남깁니다.
    """
    if u not in _UNIT_INFO_CACHE:
        toks = [t for t in kiwi.tokenize(u) if _is_noun(t)]
        _UNIT_INFO_CACHE[u] = (len(toks), any(t.tag == "NNP" for t in toks))
    return _UNIT_INFO_CACHE[u]


_REGION_GENERIC_PREFIX = re.compile(
    r"^(" + "|".join(sorted([r for r in REGION_WORDS if re.fullmatch(r"[가-힣]+", r)], key=len, reverse=True)) + r")"
    r"(맛집|카페|빵집|여행|핫플|가볼만한곳|가볼곳|맛집추천|카페추천|데이트|나들이|브런치|디저트|술집|밥집|횟집|베이커리)+")


def strip_generic_prefix(u: str) -> str:
    """'시군명+일반어' 접두를 떼어 뒤의 상호·메뉴만 남깁니다(규칙 4, SNS 전용).

    예: '전주맛집베테랑' → '베테랑', '군산카페이성당' → '이성당'. 남는 부분이 2글자 미만이면 그대로 둡니다.
    시군명 없이 '카페○○'로 시작하는 상호(예: 카페라떼, 카페온)는 상호 자체일 수 있어 떼지 않습니다.
    """
    m = _REGION_GENERIC_PREFIX.match(u)
    if m and len(u) - m.end() >= 2 and u not in USER_WORDS and u not in SYN_MAP:
        return u[m.end():]
    return u


def pretty_name(u: str) -> str:
    """표시용 이름: 한글과 섞인 4자 이하 영문 약어를 대문자로 바꿉니다(새만금ai센터 → 새만금AI센터, kcc → KCC)."""
    return re.sub(r"[a-z]{1,4}(?![a-z])", lambda m: m.group(0).upper(), u) if re.search(r"[a-z]", u) and len(re.sub(r"[^a-z]", "", u)) <= 4 else u


def strip_generic_suffix(u: str) -> str:
    """해시태그 끝의 SNS 상투어(맛집, 여행, 스타그램, 데이트 …)를 반복해서 떼어 핵심 표현을 남깁니다.

    예: '전주한옥마을데이트' → '전주한옥마을', '군산맛집추천' → '군산'(→ 지역명)
    사용자사전·통합사전에 있는 표현은 떼지 않습니다(예: '군산시간여행축제').
    """
    prev = None
    while u and u != prev and u not in USER_WORDS and u not in SYN_MAP:
        prev = u
        u = _SNS_GENERIC.sub("", u)
    return u


@lru_cache(maxsize=None)
def judge_unit(u: str, from_tag: bool = False, sns: bool = False) -> tuple:
    """정규화된 단위 하나를 (판정, 표준명칭)으로 분류합니다. 판정은 정당명/지역명/불용어/보조어/후보 중 하나입니다.

    Parameters
    ----------
    u : normalize_unit()을 거친 문자열
    from_tag : 해시태그·태그에서 온 단위면 True(SNS 상투 접미어 제거 규칙 적용)
    sns : 인스타·유튜브 단위면 True(규칙 4: 여행·맛집·카페 일반어 제외, 시군명+일반어 접두 제거)
    """
    if not u:
        return "불용어", u
    if _PARTY.search(u):
        return "정당명", u                             # 규칙 1: 정당명이 들어간 단위는 분석에서 제외
    if sns:
        core = strip_generic_prefix(u)
        if core != u:
            u = core                                   # 전주맛집베테랑 → 베테랑
    if from_tag:
        core = strip_generic_suffix(u)
        if core != u:
            if not core or core in REGION_WORDS:
                return "보조어", u                     # 예: 전주맛집, 군산여행스타그램
            u = core
    u = SYN_MAP.get(u, u)                              # 표현 통합(표준명칭)
    key = normalize_unit(u)
    if _PARTY.search(key):
        return "정당명", u
    hm = _HEAD_RE.match(key)                           # 규칙 5: 새만금 + 뒤에 오는 말
    if hm:
        rest = key[hm.end():]
        if not rest:
            return "지역명", u                          # '새만금' 단독 = 지역명
        if len(rest) >= 2 and rest not in STOP_WORDS and rest not in _HEAD_GENERIC_TAIL and rest not in REGION_WORDS \
                and not _SNS_GENERIC.fullmatch(rest) and rest not in SNS_WORDS and not _SNS_GENERIC_PLACE.fullmatch(rest):
            std = key[hm.start(2):]                    # '전북새만금국제공항' → '새만금국제공항'
            return "후보", SYN_MAP.get(std, pretty_name(std))
        return "보조어", u                              # 새만금지역, 새만금여행 등
    if key in REGION_WORDS:
        return "지역명", u
    if key.startswith("person"):
        return "불용어", u                              # 유튜브 메타데이터 찌꺼기(person홍길동)
    tm = _TITLE_NAME.fullmatch(key)
    if tm:
        return judge_unit(tm.group(2), from_tag, sns)   # '이사장최정웅', 'NEWS이정민' → 이름만
    if _ADMIN_UNIT.fullmatch(key):
        return "지역명", u                              # 장수군장수읍, 전북순창군순창읍, 봉동읍, 김제시금산면
    if len(key) < 2 or key.isdigit() or key in STOP_WORDS or len(key) > CONFIG["MAX_UNIT_LEN"]:
        return "불용어", u
    if re.fullmatch(r"제?\d+[가-힣]{0,2}", key):                 # 제1, 1층, 2학기, 3인
        return "불용어", u
    if re.fullmatch(r"[0-9a-z]+", key):                         # 영문·숫자만: 화이트리스트·사용자사전에 있을 때만 후보
        return ("후보", u) if (key in WHITELIST or key in USER_WORDS or key in SYN_STD) else ("보조어", u)
    _rest = key                                                 # 지역명 접두어를 최대 3번까지 떼어 냄(전북+전주시+완산구)
    for _ in range(3):
        if not _REGION_PREFIX.match(_rest):
            break
        _nxt = _REGION_PREFIX.sub("", _rest, count=1).lstrip("시군")
        if not _nxt or _nxt == _rest:
            _rest = "" if not _nxt else _rest
            break
        _rest = _nxt
        if _rest in REGION_WORDS:
            break
    if _rest == "":
        return "지역명", u                                      # 지역명+시/군만으로 된 단위(전북시 등)
    if (_SUBREGION.fullmatch(key) and (not key.endswith(("동", "읍", "면")) or unit_token_info(key)[1])) or \
       (_rest != key and (_rest in REGION_WORDS or _SUBREGION.fullmatch(_rest))):
        return "지역명", u                                      # 하위 행정구역(완산구, 전주시완산구, 전북장수군)
    if key in AUX_WORDS:
        return "보조어", u
    if sns and (key in SNS_WORDS or _SNS_GENERIC_PLACE.fullmatch(key)):
        return "보조어", u                              # 규칙 4: 감성카페, 빵지순례, 소품샵 …
    m = _REGION_PREFIX.match(key)                     # 시군명 + 일반어 (전주맛집, 익산청년, 김제시민)
    if m and m.end() < len(key):
        rest = key[m.end():]
        if rest in REGION_WORDS or rest in _PARTICLES:
            return "지역명", u                                  # 전북전주시, 전라북도장수군, 순창군이, 익산시가
        if rest in AUX_WORDS or rest in STOP_WORDS or _SNS_GENERIC.fullmatch(rest) or rest in {"시", "군", "청", "도"}:
            return "보조어", u
        if sns and (rest in SNS_WORDS or _SNS_GENERIC_PLACE.fullmatch(rest)):
            return "보조어", u                          # 전주감성카페, 군산빵지순례
    if key in WHITELIST or key in USER_WORDS or key in SYN_STD:
        return "후보", (u if u in SYN_MAP.values() else pretty_name(u))
    if len(key) <= 2:
        n_tok, has_nnp = unit_token_info(key)
        if n_tok <= 1 and not has_nnp:
            return "보조어", u                             # 2글자 단일 일반명사(예산, 교육, 행사 …)
    return "후보", pretty_name(u)


for _u, _f, _s in [("전주맛집", True, True), ("전주한옥마을데이트", True, True), ("군산시간여행축제", False, False),
                   ("예산", False, False), ("전주시", False, False), ("새만금", False, False), ("새만금국제공항", False, False),
                   ("새만금AI센터", False, False), ("더불어민주당", False, False), ("감성카페", True, True),
                   ("전주맛집베테랑", True, True), ("익산청년", False, False), ("지평선축제", False, False)]:
    print(f"{_u:14s} → {judge_unit(normalize_unit(_u), _f, _s)}")

In [ ]:
def select_ngrams(units: pd.DataFrame, media: str) -> set:
    """어절을 잇는 복합명사 후보 중 '자주 함께 나오고 결합이 강한' 것만 고릅니다.

    기준(둘 다 충족):
      ① df(AB) ≥ NGRAM_MIN_DF[media]                 … AB가 등장한 콘텐츠 수
      ② df(AB) / min(df(A), df(B)) ≥ NGRAM_ASSOC      … 구성 단어 중 덜 흔한 쪽이 AB 안에서 쓰이는 비율
    그리고 마지막 구성어가 '추진·개최·지원' 같은 동작성 일반어이거나 '시장·군수·의원' 같은 직함이면
    (예: '우범기 전주시장' → 인물과 직함을 기계적으로 이은 것) 제외하고, 구성어 가운데 불용어가 있어도 제외합니다.

    규칙 5(새만금): 첫 구성어가 REGION_COMPOUND_HEADS(새만금)이면 결합 강도 ②를 보지 않고
    df(AB) ≥ REGION_COMPOUND_MIN_DF[media]만 확인합니다. '새만금'은 어디에나 붙는 지역명이라 결합 강도가 늘 낮게 나오지만,
    '새만금 국제공항', '새만금 이차전지'처럼 뒤에 오는 말과 묶어야 구체적인 대상이 되기 때문입니다.
    """
    df_unit, df_ng = Counter(), Counter()
    for runs, ngs in zip(units["runs"], units["ngrams"]):
        df_unit.update({normalize_unit(r) for r in runs})
        df_ng.update({tuple(normalize_unit(x) for x in ng) for ng in ngs})
    keep = set()
    min_df, assoc = CONFIG["NGRAM_MIN_DF"][media], CONFIG["NGRAM_ASSOC"]
    heads, hmin = set(_HEADS), CONFIG["REGION_COMPOUND_MIN_DF"][media]
    for ng, c in df_ng.items():
        if any(not x for x in ng):
            continue
        if ng[0] in heads:                                         # 새만금 + 뒤에 오는 말
            if c >= hmin and not any(x in heads for x in ng[1:]) and ng[-1] not in _VERBAL_TAIL \
                    and not _TITLE_TAIL.search(ng[-1]) and not any(x in STOP_WORDS for x in ng[1:]):
                keep.add(ng)
            continue
        if c < min_df:
            continue
        if ng[-1] in _VERBAL_TAIL or _TITLE_TAIL.search(ng[-1]) or any(x in STOP_WORDS for x in ng):
            continue
        if c / max(1, min(df_unit[x] for x in ng)) >= assoc:
            keep.add(ng)
    return keep


def build_content_targets(df: pd.DataFrame, units: pd.DataFrame, media: str):
    """콘텐츠별로 관심대상 후보(표준명칭) 집합을 만들어 '콘텐츠×관심대상' 긴 표(long table)로 반환합니다.

    한 콘텐츠 안에서 같은 대상이 제목·설명·태그·타임라인에 여러 번 나와도 1건만 남깁니다(의뢰서 4.3).
    보조어·지역명은 따로 모아 보조 집계용 표로 돌려줍니다.

    Returns
    -------
    ct  : DataFrame[content_id, target, src]      … 후보(src: 본문/복합/태그)
    aux : DataFrame[content_id, target, judge]     … 보조어·지역명
    mapping : Counter{(원표현, 표준명칭): 건수}      … 표현 통합 대응 기록
    """
    keep_ng = select_ngrams(units, media)
    tag_map = dict(zip(df["content_id"], df["tag_list"]))
    sns = media in ("인스타", "유튜브")
    rows, aux_rows, mapping = [], [], Counter()
    for cid, runs, ngs in zip(units["content_id"], units["runs"], units["ngrams"]):
        seen = {}
        cand = [(r, "본문") for r in runs]
        kept = [tuple(normalize_unit(x) for x in ng) for ng in ngs]
        kept = [ng for ng in kept if ng in keep_ng]
        # 새만금 복합어는 가장 긴 것만 남김(새만금 일자리 창출이 있으면 '새만금일자리'는 뺌)
        kept = [ng for ng in kept if not (ng[0] in _HEADS and any(len(o) > len(ng) and o[:len(ng)] == ng for o in kept))]
        cand += [("".join(ng), "복합") for ng in kept]
        cand += [(t, "태그") for t in tag_map.get(cid, [])]
        for raw, src in cand:
            u = normalize_unit(raw)
            judge, std = judge_unit(u, from_tag=(src == "태그"), sns=sns)
            if judge == "후보":
                if u != normalize_unit(std):
                    mapping[(u, std)] += 1
                if std not in seen:
                    seen[std] = src
            elif judge in ("보조어", "지역명", "정당명"):
                aux_rows.append((cid, std, judge))
        rows += [(cid, t, s) for t, s in seen.items()]
    ct = pd.DataFrame(rows, columns=["content_id", "target", "src"])
    aux = pd.DataFrame(aux_rows, columns=["content_id", "target", "judge"]).drop_duplicates()
    return ct, aux, mapping


CT, AUX, MAPPING = {}, {}, Counter()
for m, d in zip(MEDIA, (news, ig, yt)):
    t0 = time.time()
    CT[m], AUX[m], mp = build_content_targets(d[d["valid"]], UNITS[m], m)
    MAPPING.update(mp)
    print(f"{m}: 후보 출현 {len(CT[m]):,}행, 고유 후보 {CT[m]['target'].nunique():,}개 ({time.time()-t0:.0f}초)")

In [ ]:
def attach_meta(ct: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    """콘텐츠×관심대상 표에 날짜·연·월·시군·출처·빈도 산정 단위를 붙입니다."""
    meta_cols = ["content_id", "count_key", "date", "year", "month", "sigun_all", "source", "type"]
    if "collect_mode" in df:
        meta_cols.append("collect_mode")
    return ct.merge(df[meta_cols], on="content_id", how="left")


for m, d in zip(MEDIA, (news, ig, yt)):
    CT[m] = attach_meta(CT[m], d)
    AUX[m] = attach_meta(AUX[m], d)


def collapse_source_boilerplate(ct: pd.DataFrame, df: pd.DataFrame, media: str):
    """같은 채널(유튜브)·작성자(인스타)가 거의 모든 콘텐츠에 반복해 붙이는 대상은 그 출처에서 *1건만* 셉니다.

    예: 휴대폰 수리점 계정이 200여 개 게시물마다 같은 해시태그 묶음('액정수리', '데이터복구' …)을 붙이거나,
    부동산 채널이 영상마다 같은 설명란 문구('급매물', '권리분석' …)를 넣으면 그 단어가 상위에 오릅니다.
    이런 반복 생산은 여러 이용자의 관심이 아니므로, 출처 콘텐츠가 SOURCE_BOILER_MIN건 이상이고
    대상이 그 출처 콘텐츠의 SOURCE_BOILER_SHARE(30건 이상이면 _BIG) 이상에 나오면 가장 이른 1건만 남깁니다.
    다른 출처의 언급은 그대로 셉니다. 뉴스(언론사 6곳)에는 적용하지 않습니다.

    Returns
    -------
    ct  : 반복분을 뺀 콘텐츠×관심대상 표
    log : DataFrame[매체, 출처, 출처콘텐츠수, 관심대상, 원래건수, 반영건수]
    """
    cols = ["매체", "출처", "출처콘텐츠수", "관심대상", "원래건수", "반영건수"]
    if media == "뉴스":
        return ct, pd.DataFrame(columns=cols)
    n_src = df[df["valid"]].groupby("source")["content_id"].nunique()
    pair = ct.groupby(["source", "target"])["content_id"].nunique()
    src_n = n_src.reindex(pair.index.get_level_values(0)).values
    thr = np.where(src_n >= 30, CONFIG["SOURCE_BOILER_SHARE_BIG"], CONFIG["SOURCE_BOILER_SHARE"])
    flag = pair[(src_n >= CONFIG["SOURCE_BOILER_MIN"]) & (pair.values / src_n >= thr) & (pair.values >= 2)]
    if flag.empty:
        return ct, pd.DataFrame(columns=cols)
    c = ct.sort_values("date")
    is_flag = pd.MultiIndex.from_frame(c[["source", "target"]]).isin(flag.index)
    keep = ~is_flag | ~c.duplicated(["source", "target"])
    log = flag.rename("원래건수").reset_index()
    log.columns = ["출처", "관심대상", "원래건수"]
    log.insert(0, "매체", media)
    log.insert(2, "출처콘텐츠수", log["출처"].map(n_src))
    log["반영건수"] = 1
    return c[keep].sort_index(), log.sort_values("원래건수", ascending=False)[cols]


CT["유튜브"] = CT["유튜브"][~CT["유튜브"]["target"].isin(YT_STAFF)].reset_index(drop=True)   # 본문에 남은 제작진 이름 제외

SRC_BOILER = []
for m, d in zip(MEDIA, (news, ig, yt)):
    before = len(CT[m])
    CT[m], lg = collapse_source_boilerplate(CT[m], d, m)
    SRC_BOILER.append(lg)
    if m != "뉴스":
        print(f"{m}: 반복 생산 상쇄 {len(lg):,}쌍(출처×대상), 줄어든 행 {before - len(CT[m]):,}")
SRC_BOILER = pd.concat(SRC_BOILER, ignore_index=True)

# 규칙 1: 정당명 제외 기록(매체별 제외된 표현과 콘텐츠 수)
PARTY_EXCLUDED = pd.concat([
    AUX[m][AUX[m]["judge"] == "정당명"].groupby("target")["count_key"].nunique().rename("제외 콘텐츠 수")
       .reset_index().assign(매체=m) for m in MEDIA], ignore_index=True)
PARTY_EXCLUDED = PARTY_EXCLUDED.rename(columns={"target": "제외 표현"})[["매체", "제외 표현", "제외 콘텐츠 수"]] \
    .sort_values(["매체", "제외 콘텐츠 수"], ascending=[True, False])
PARTY_EXCLUDED["제외 사유"] = "정당명(분석 제외)"
print("정당명이 나와 제외 처리된 콘텐츠 수:",
      {m: int(AUX[m].loc[AUX[m]["judge"] == "정당명", "count_key"].nunique()) for m in MEDIA})
display(SRC_BOILER.head(15))

def _area_rule_hit(target: str) -> str:
    """영역분류규칙.csv의 첫 번째로 맞는 영역(없으면 빈 문자열). 범용어 판정에서만 쓰는 간단한 버전입니다."""
    key = normalize_unit(target)
    for area, pat in _AREA_RULES_SIMPLE:
        if pat.search(key):
            return area
    return ""


_AREA_RULES_SIMPLE = [(r["영역"], re.compile(r["패턴"], re.I))
                      for _, r in pd.read_csv(DICT_DIR / "영역분류규칙.csv").sort_values("우선순위").iterrows()]


_SURNAMES = set("김이박최정강조윤장임한오서신권황안송류유홍전고문양손배백허남심노하곽성차주우구민진나지엄채원천방공현함변염여추도소석선설마길연위표명기반왕금옥육인맹제모탁국어은편용예경봉")
_TITLE_AFTER = (r"\s?(전북|전북도|전북특별자치도|[가-힣]{2}(시|군)|[가-힣]{2})?\s?(국회의원|의원|시장|군수|도지사|지사|교육감|후보|예비후보|"
                r"위원장|의장|대표|당선인|당선자|전\s?의원|전\s?시장|부지사|부시장|부군수|원내대표|최고위원|장관|총장|대통령|"
                r"전\s?대통령|여사|총리|청장|이사장|은행장|본부장|원장|사장|회장|교수|변호사|감독|선수)")


_PERSON_STATS = {}


def _person_title_counts():
    """뉴스 전체를 *한 번만* 훑어 '3~4글자 한글 + (지역) + 직함' 패턴의 앞 단어별 건수와 단어별 등장 기사 수를 셉니다."""
    if not _PERSON_STATS:
        pat = re.compile(r"(?<![가-힣])([가-힣]{2,3})" + _TITLE_AFTER)
        title_cnt = Counter()
        for t in news.loc[news["valid"], "text"]:
            title_cnt.update({m.group(1) for m in pat.finditer(t)})
        _PERSON_STATS["title"] = title_cnt
        _PERSON_STATS["df"] = CT["뉴스"].groupby("target")["count_key"].nunique()
    return _PERSON_STATS["title"], _PERSON_STATS["df"]


def looks_like_person(word: str, min_ratio: float = 0.2) -> bool:
    """3~4글자 한글이고 성씨로 시작하는 단어가 뉴스에서 직함(의원·시장·군수·후보 등) 앞에 자주 오면 인명으로 봅니다.

    Kiwi는 사전에 없는 인명을 일반명사(NNG)로 분석하므로, 범용어 판정에서 인명이 빠지지 않도록 하는 보호 규칙입니다.
    판정 기준: (단어 + 직함 형태가 나온 기사 수) ÷ (단어가 나온 기사군 수) ≥ min_ratio, 그리고 직함 형태 5건 이상.
    """
    if not re.fullmatch(r"[가-힣]{2,3}", word) or word[0] not in _SURNAMES:
        return False
    key = normalize_unit(word)
    if key in WHITELIST or key in USER_WORDS or key in SYN_STD or _area_rule_hit(word):
        return False                                  # 특산품·지명·축제처럼 이미 정해진 대상(반딧불 등)은 인명이 아님
    title_cnt, df = _person_title_counts()
    n_title, n = title_cnt.get(word, 0), df.get(word, 0)
    return n_title >= 5 and n > 0 and n_title / n >= min_ratio


def detect_spread_generic(min_n=30, min_cover=10, min_entropy=0.8) -> pd.DataFrame:
    """14개 시군에 고르게 퍼진 *일반명사*를 범용어(보조어)로 판정합니다.

    '관광객', '지역사회', '마무리'처럼 어느 지역에서나 쓰이는 말은 단어 목록으로 다 막기 어렵습니다.
    그래서 세 매체를 합친 시군 분포로 판단합니다. 조건은 아래와 같습니다(모두 충족 시 범용어).
      ① 등장 콘텐츠 min_n건 이상
      ② min_cover개 이상 시군에서 등장('전북 공통' 제외)
      ③ 시군 분포의 정규화 엔트로피 ≥ min_entropy(1에 가까울수록 고르게 퍼짐)
      ④ 형태소 분석상 고유명사(NNP)가 없고, 화이트리스트·사용자사전·통합사전에 없으며,
         영역 규칙상 상품·장소·문화·기업 대상으로 분류되지 않음(예: '반도체', '가을축제'는 유지)
    '한옥마을'(전주 집중), '김관영'(고유명사)은 남고 '관광객', '지역경제'는 빠집니다.
    """
    allct = pd.concat([CT[m][["target", "content_id", "sigun_all"]] for m in MEDIA])
    e = allct.explode("sigun_all")
    e = e[e["sigun_all"].isin(SIGUN)]
    dist = e.groupby(["target", "sigun_all"])["content_id"].nunique().unstack(fill_value=0)
    n = allct.groupby("target")["content_id"].nunique()
    p = dist.div(dist.sum(axis=1), axis=0)
    ent = -(p * np.log(p.where(p > 0, 1))).sum(axis=1) / np.log(len(SIGUN))
    cover = (dist > 0).sum(axis=1)
    cand = pd.DataFrame({"n": n, "시군수": cover, "엔트로피": ent}).dropna()
    cand = cand[(cand["n"] >= min_n) & (cand["시군수"] >= min_cover) & (cand["엔트로피"] >= min_entropy)]
    keep_areas = {"상품·먹거리", "장소·공간", "문화·콘텐츠·스포츠", "기업·산업"}
    rows = []
    for t, r in cand.iterrows():
        key = normalize_unit(t)
        if key in WHITELIST or key in USER_WORDS or key in SYN_STD:
            continue
        if unit_token_info(key)[1]:                 # 고유명사 포함
            continue
        if _REGION_PREFIX.match(key):               # 지역명으로 시작하는 기관·대상(전북농협 등)
            continue
        if looks_like_person(t):                    # 형태소 분석기가 모르는 인명(이원택 등)
            continue
        if _area_rule_hit(t) in keep_areas:
            continue
        rows.append((t, int(r["n"]), int(r["시군수"]), round(r["엔트로피"], 3)))
    return pd.DataFrame(rows, columns=["범용어", "등장건수", "등장시군수", "정규화엔트로피"]).sort_values("등장건수", ascending=False)


SPREAD_GENERIC = detect_spread_generic()
_gen = set(SPREAD_GENERIC["범용어"])
for m in MEDIA:
    moved = CT[m][CT[m]["target"].isin(_gen)].drop(columns="src").assign(judge="보조어")
    AUX[m] = pd.concat([AUX[m], moved], ignore_index=True)
    CT[m] = CT[m][~CT[m]["target"].isin(_gen)].reset_index(drop=True)
print(f"범용어로 판정해 보조어로 옮긴 대상: {len(_gen)}개")
print("예:", ", ".join(SPREAD_GENERIC["범용어"].head(40)))

# 표현 통합 대응표(원표현 → 표준명칭, 적용 건수): 산출물 ②에 포함
MAPPING_TABLE = pd.DataFrame([(a, b, c) for (a, b), c in MAPPING.items()],
                             columns=["원표현(정규화)", "표준명칭", "적용 건수"]).sort_values("적용 건수", ascending=False)
MAPPING_TABLE.head(20)

## 7. 빈도·비중 집계

- **등장 건수** = 그 대상이 나온 *고유 콘텐츠 수*(뉴스는 기사군 수)
- **등장 비중** = 등장 건수 ÷ 같은 매체·기간(·시군)의 유효 콘텐츠 수(분모)
- 한 콘텐츠에 여러 대상이 나오면 각 대상에 1건씩 세므로, 대상별 합계는 전체 콘텐츠 수와 다릅니다.

In [ ]:
BASE = {m: d[d["valid"]] for m, d in zip(MEDIA, (news, ig, yt))}


def in_period(s: pd.Series, start, end) -> pd.Series:
    """날짜 Series가 [start, end] 구간(양 끝 포함, 종료일은 그날 23:59까지)에 드는지 판정합니다."""
    return (s >= pd.Timestamp(start)) & (s < pd.Timestamp(end) + pd.Timedelta(days=1))


def denominator(media: str, start=None, end=None, sigun=None, trend=False) -> int:
    """분모(유효 콘텐츠 수)를 계산합니다.

    trend=True이고 유튜브이면 '기본' 수집분만 셉니다(연도 비교용, CONFIG['YT_TREND_BASIS_ONLY']).
    sigun을 주면 그 시군이 시군 목록에 들어 있는 콘텐츠만 셉니다.
    """
    b = BASE[media]
    if start is not None:
        b = b[in_period(b["date"], start, end)]
    if trend and media == "유튜브" and CONFIG["YT_TREND_BASIS_ONLY"]:
        b = b[b["collect_mode"] == "기본"]
    if sigun is not None:
        b = b[b["sigun_all"].map(lambda L: sigun in L)]
    return b["count_key"].nunique()


def subset_ct(media: str, start=None, end=None, trend=False, table=None) -> pd.DataFrame:
    """콘텐츠×관심대상 표를 기간(·유튜브 기본 수집분)으로 거릅니다."""
    c = (table or CT)[media]
    if start is not None:
        c = c[in_period(c["date"], start, end)]
    if trend and media == "유튜브" and CONFIG["YT_TREND_BASIS_ONLY"]:
        c = c[c["collect_mode"] == "기본"]
    return c


def freq_table(media: str, start=None, end=None, trend=False, table=None) -> pd.DataFrame:
    """기간 하나의 관심대상별 등장 건수·비중·순위·출처 다양성을 계산합니다.

    Returns
    -------
    DataFrame[target, 등장건수, 등장비중, 순위, 출처다양성, 분모]
    출처다양성: 뉴스=언론사 수, 인스타=작성자 수, 유튜브=채널 수
    """
    c = subset_ct(media, start, end, trend, table)
    n = denominator(media, start, end, trend=trend)
    g = c.groupby("target").agg(등장건수=("count_key", "nunique"), 출처다양성=("source", "nunique"))
    g["등장비중"] = g["등장건수"] / n
    g["분모"] = n
    g = g.sort_values(["등장건수", "출처다양성"], ascending=False)
    g["순위"] = np.arange(1, len(g) + 1)
    return g.reset_index()[["target", "등장건수", "등장비중", "순위", "출처다양성", "분모"]]


# 공통 관측기간 기준 매체별 상위 관심대상
FREQ_COMMON = {m: freq_table(m, CONFIG["COMMON_START"], CONFIG["COMMON_END"]) for m in MEDIA}
pd.concat({m: FREQ_COMMON[m].head(25)[["target", "등장건수", "등장비중"]].reset_index(drop=True) for m in MEDIA}, axis=1)

In [ ]:
def yearly_table(media: str, years=(2023, 2024, 2025, 2026), top_n=200) -> pd.DataFrame:
    """연도별 등장 건수·비중·순위를 한 표로 만듭니다(넓은 표).

    연도는 자료별 실제 수록 범위를 따릅니다(2026은 종료일까지). 2023년 SNS 1~10월은 수집 방식이 다른 보완 수집분입니다.
    연도 간 증감은 9단계의 '과거 동기간' 비교를 기준으로 보고, 이 표는 순위·비중 흐름을 보는 데 씁니다.
    """
    parts = {}
    for y in years:
        f = freq_table(media, f"{y}-01-01", f"{y}-12-31", trend=True).set_index("target")
        parts[y] = f[["등장건수", "등장비중", "순위"]]
    wide = pd.concat(parts, axis=1)
    wide.columns = [f"{y}_{c}" for y, c in wide.columns]
    wide["전체건수"] = wide[[c for c in wide.columns if c.endswith("등장건수")]].sum(axis=1)
    return wide.sort_values("전체건수", ascending=False).head(top_n)


YEARLY = {m: yearly_table(m) for m in MEDIA}
YEARLY["뉴스"].head(15)

In [ ]:
def sigun_table(media: str, top_n=20) -> pd.DataFrame:
    """시군별 상위 관심대상(공통 관측기간). 여러 시군에 걸친 콘텐츠는 각 시군에 1번씩 셉니다.

    비중의 분모는 '그 시군이 시군 목록에 들어 있는 유효 콘텐츠 수'입니다.
    """
    c = subset_ct(media, CONFIG["COMMON_START"], CONFIG["COMMON_END"]).explode("sigun_all")
    out = []
    for sg, grp in c.groupby("sigun_all"):
        n = denominator(media, CONFIG["COMMON_START"], CONFIG["COMMON_END"], sigun=sg)
        f = grp.groupby("target")["count_key"].nunique().sort_values(ascending=False).head(top_n)
        out.append(pd.DataFrame({"매체": media, "시군": sg, "순위": range(1, len(f) + 1), "관심대상": f.index,
                                 "등장건수": f.values, "등장비중": f.values / max(n, 1), "분모": n}))
    return pd.concat(out, ignore_index=True)


SIGUN_TOP = pd.concat([sigun_table(m) for m in MEDIA], ignore_index=True)
SIGUN_TOP[(SIGUN_TOP["시군"] == "군산시") & (SIGUN_TOP["순위"] <= 10)].pivot(index="순위", columns="매체", values="관심대상")

In [ ]:
def monthly_series(media: str, targets: list, trend=True) -> pd.DataFrame:
    """관심대상별 월간 등장 비중(그 달 유효 콘텐츠 대비) 표를 만듭니다. 행=월, 열=관심대상."""
    c = subset_ct(media, CONFIG["COMMON_START"], CONFIG["COMMON_END"], trend=trend)
    c = c[c["target"].isin(targets)]
    num = c.groupby(["month", "target"])["count_key"].nunique().unstack(fill_value=0)
    b = BASE[media]
    b = b[in_period(b["date"], CONFIG["COMMON_START"], CONFIG["COMMON_END"])]
    if trend and media == "유튜브" and CONFIG["YT_TREND_BASIS_ONLY"]:
        b = b[b["collect_mode"] == "기본"]
    den = b.groupby("month")["count_key"].nunique()
    return num.reindex(den.index, fill_value=0).div(den, axis=0)


def plot_monthly(media: str, top=8):
    """상위 관심대상의 월간 등장 비중을 선그래프로 그립니다."""
    tg = FREQ_COMMON[media]["target"].head(top).tolist()
    ms = monthly_series(media, tg)
    ax = (ms * 100).plot(figsize=(12, 4), title=f"{media}: 상위 {top}개 관심대상 월간 등장 비중(%)")
    ax.set_xlabel(""); ax.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()


for m in MEDIA:
    plot_monthly(m)

In [ ]:
def aux_table(media: str, top_n=50) -> pd.DataFrame:
    """보조어(추상·일반어, 시군명+일반어)의 등장 건수. 주요 순위에서는 뺐지만 맥락 참고용으로 제공합니다."""
    a = AUX[media]
    a = a[(a["judge"] == "보조어") & in_period(a["date"], CONFIG["COMMON_START"], CONFIG["COMMON_END"])]
    f = a.groupby("target")["count_key"].nunique().sort_values(ascending=False).head(top_n)
    return pd.DataFrame({"매체": media, "보조어": f.index, "등장건수": f.values})


AUX_TOP = pd.concat([aux_table(m) for m in MEDIA], ignore_index=True)
pd.concat({m: AUX_TOP[AUX_TOP["매체"] == m].head(10)[["보조어", "등장건수"]].reset_index(drop=True) for m in MEDIA}, axis=1)

## 8. 6개 영역 1차 분류와 검토표

**방법.** `dict/영역분류규칙.csv`의 정규식(접미어·키워드)으로 1차 자동 분류합니다. 우선순위가 높은 규칙이 **주분류**, 다음으로 맞은 다른 영역이 **보조분류**가 됩니다. 어느 규칙에도 맞지 않으면 `미분류(검토필요)`로 둡니다.

**검토 방식.** 의뢰서 5장에 따라 *단어가 아니라 실제 문맥*으로 분류해야 하므로, 검토표에 **대표 콘텐츠 3건의 제목·본문 발췌**를 함께 넣었습니다. 검토자가 `dict/영역분류_수기.csv`에 `관심대상, 주분류, 보조분류, 메모`를 적고 이 단계부터 다시 실행하면 수기 분류가 자동 분류보다 우선 적용됩니다.

In [ ]:
AREAS = ["상품·먹거리", "장소·공간", "문화·콘텐츠·스포츠", "기업·산업", "정책·공공사업", "인물·기관·사회 현상"]
AREA_RULES = pd.read_csv(DICT_DIR / "영역분류규칙.csv").sort_values("우선순위")
AREA_RULES["re"] = AREA_RULES["패턴"].map(lambda p: re.compile(p, re.I))
MANUAL = pd.read_csv(DICT_DIR / "영역분류_수기.csv").dropna(subset=["관심대상"])
MANUAL_MAP = {normalize_unit(r["관심대상"]): (r["주분류"], r.get("보조분류", ""), "수기 분류: " + str(r.get("메모", ""))) for _, r in MANUAL.iterrows()}


@lru_cache(maxsize=None)
def classify_area(target: str) -> tuple:
    """관심대상 하나를 (주분류, 보조분류, 분류근거, 검토필요)로 분류합니다.

    수기 분류표에 있으면 그것을 따르고, 없으면 규칙 순서대로 맞춰 봅니다.
    분류근거에는 맞은 규칙의 영역과 실제로 일치한 문자열을 적어, 검토자가 판단 근거를 확인할 수 있게 합니다.
    """
    if normalize_unit(target) in MANUAL_MAP:
        a, b, why = MANUAL_MAP[normalize_unit(target)]
        return a, (b if isinstance(b, str) else ""), why, False
    key = normalize_unit(target)
    hits = []
    for _, r in AREA_RULES.iterrows():
        m = r["re"].search(key)
        if m and r["영역"] not in [h[0] for h in hits]:
            hits.append((r["영역"], m.group(0)))
    if not hits:
        if looks_like_person(target):
            return "인물·기관·사회 현상", "", "인명 판정(뉴스에서 직함과 함께 쓰임)", False
        return "미분류(검토필요)", "", "규칙 불일치", True
    main, sub = hits[0], (hits[1] if len(hits) > 1 else ("", ""))
    why = f"'{main[1]}' → {main[0]}" + (f" / '{sub[1]}' → {sub[0]}" if sub[0] else "")
    return main[0], sub[0], why, len(hits) > 1          # 두 영역 이상 맞으면 검토 권장


def sample_contexts(media: str, target: str, k=3) -> str:
    """관심대상이 나온 대표 콘텐츠 k건의 날짜와 발췌문을 이어 붙입니다(검토자 문맥 확인용)."""
    ids = CT[media].loc[CT[media]["target"] == target, "content_id"].head(400)
    b = BASE[media].set_index("content_id").loc[ids]
    if media == "뉴스":
        b = b.sort_values("dup_size", ascending=False)       # 여러 언론사가 다룬 기사 우선
    elif "views" in b:
        b = b.sort_values("views", ascending=False, na_position="last")
    out = []
    for _, r in b.head(k).iterrows():
        txt = (r["title"] or r["body"] or "").replace("\n", " ")
        out.append(f"[{r['date']:%Y-%m-%d}] {txt[:70]}")
    return " ‖ ".join(out)


def build_review_pool() -> pd.DataFrame:
    """매체별 상위 TOP_N_REVIEW개 후보를 모아 영역 분류 검토표를 만듭니다."""
    rows = []
    for m in MEDIA:
        for _, r in FREQ_COMMON[m].head(CONFIG["TOP_N_REVIEW"]).iterrows():
            rows.append((r["target"], m, r["등장건수"], r["순위"]))
    pool = pd.DataFrame(rows, columns=["관심대상", "매체", "등장건수", "순위"])
    wide = pool.pivot_table(index="관심대상", columns="매체", values="순위", aggfunc="min")
    wide.columns = [f"{c}_순위" for c in wide.columns]
    wide = wide.reset_index()
    cls = wide["관심대상"].map(classify_area)
    wide["주분류"] = [c[0] for c in cls]
    wide["보조분류"] = [c[1] for c in cls]
    wide["분류근거"] = [c[2] for c in cls]
    wide["검토필요"] = [c[3] for c in cls]
    best_media = wide[[f"{m}_순위" for m in MEDIA if f"{m}_순위" in wide]].idxmin(axis=1).str.replace("_순위", "")
    wide["대표문맥"] = [sample_contexts(bm, t) for bm, t in zip(best_media, wide["관심대상"])]
    return wide.sort_values([f"{m}_순위" for m in MEDIA], na_position="last")


t0 = time.time()
AREA_REVIEW = build_review_pool()
# 영역 분류 범위: 검토표(상위 TOP_N_REVIEW)보다 넓게, 매체별 상위 AREA_CLASSIFY_TOP개 후보 전체를 분류합니다.
_area_pool = set().union(*[set(FREQ_COMMON[m]["target"].head(CONFIG["AREA_CLASSIFY_TOP"])) for m in MEDIA])
AREA_MAP = {t: classify_area(t)[0] for t in _area_pool}
print(f"검토표 {len(AREA_REVIEW)}개 후보, 영역 분류 대상 {len(AREA_MAP):,}개 ({time.time()-t0:.0f}초)")
display(AREA_REVIEW["주분류"].value_counts())
AREA_REVIEW.head(15)

In [ ]:
def merge_suggestions(targets: list) -> pd.DataFrame:
    """표현 통합 *제안*(자동 통합 아님): 'X'와 '시군명+X'가 모두 후보에 있으면 같은 대상일 가능성을 표시합니다.

    예: '한옥마을' ↔ '전주한옥마을'. 실제로 다른 대상일 수 있으므로 검토 후 dict/표현통합사전.csv에 추가합니다.
    """
    tset = set(targets)
    rows = []
    for t in targets:
        m = _REGION_PREFIX.match(normalize_unit(t))
        if m and m.end() < len(t):
            rest = t[m.end():]
            if rest in tset and len(rest) >= 2:
                rows.append((rest, t, "시군명 접두 변형"))
    return pd.DataFrame(rows, columns=["표현A", "표현B", "근거"]).drop_duplicates()


MERGE_SUGGEST = merge_suggestions(AREA_REVIEW["관심대상"].tolist())
MERGE_SUGGEST.head(20)

### 8.1 6개 영역별 다빈도 키워드(매체별, 복합명사 중심)

공통 관측기간(2023-01-01~2026-10-26) 등장 건수로 **매체마다 6개 영역별 상위 `AREA_TOP_N`개**를 뽑습니다.
- **복합명사 중심:** 명사 2개 이상이 결합한 말(전주비빔밥, 새만금국제공항, 임실치즈테마파크), 고유명사, 인명, 화이트리스트에 있는 특산품·메뉴만 남기고 '전망대', '박물관'처럼 *단일 일반명사*는 뺍니다. 상품·먹거리 영역의 메뉴명(떡볶이, 물짜장 등)은 단일 명사라도 구체적인 대상이므로 남깁니다.
- 정당명은 6단계에서 이미 제외했습니다. SNS의 여행·맛집·카페 일반어도 6단계에서 보조어로 뺐습니다.

In [ ]:
def is_specific(target: str) -> bool:
    """복합명사 중심 목록에 넣을 '구체적인 대상'인지 판정합니다.

    다음 중 하나면 True: 명사 토큰 2개 이상(복합명사), 고유명사 포함, 화이트리스트·사용자사전·표현통합사전에 있음,
    인명으로 판정됨, 4글자 이상, 영역이 상품·먹거리(메뉴·특산품).
    """
    key = normalize_unit(target)
    n_tok, has_nnp = unit_token_info(key)
    return (n_tok >= 2 or has_nnp or key in WHITELIST or key in USER_WORDS or key in SYN_STD
            or len(key) >= 4 or looks_like_person(target) or AREA_MAP.get(target) == "상품·먹거리")


def area_top_keywords(media: str, n: int) -> pd.DataFrame:
    """매체 하나의 6개 영역별 다빈도 키워드 표(긴 표).

    Returns
    -------
    DataFrame[매체, 영역, 영역내순위, 관심대상, 등장건수, 등장비중(%), 출처다양성, 매체전체순위, 2026비중(%), 과거비중(%)]
    과거비중 = 2023~2025년 같은 기간(1/1~10/26) 합산 기준, 2026비중 = 2026-01-01~10-26 기준
    """
    f = FREQ_COMMON[media].copy()
    f["영역"] = f["target"].map(lambda t: AREA_MAP.get(t) or classify_area(t)[0])
    f = f[f["영역"].isin(AREAS)]
    f = f[f["target"].map(is_specific)]
    f = f.groupby("영역", sort=False).head(n).copy()
    f["영역내순위"] = f.groupby("영역").cumcount() + 1
    cur = freq_table(media, *CONFIG["CUR_PERIOD"], trend=True).set_index("target")["등장비중"]
    past_parts = [freq_table(media, f"{y}-01-01", f"{y}{CONFIG['CUR_PERIOD'][1][4:]}", trend=True).set_index("target")
                  for y in CONFIG["PAST_YEARS"]]
    past_n = sum(p["등장건수"].reindex(f["target"]).fillna(0) for p in past_parts)
    past_den = sum(p["분모"].iloc[0] for p in past_parts)
    out = pd.DataFrame({
        "매체": media, "영역": f["영역"].values, "영역내순위": f["영역내순위"].values, "관심대상": f["target"].values,
        "등장건수": f["등장건수"].values, "등장비중(%)": (f["등장비중"] * 100).round(3).values,
        "출처다양성": f["출처다양성"].values, "매체전체순위": f["순위"].values,
        "2026비중(%)": (cur.reindex(f["target"]).fillna(0) * 100).round(3).values,
        "과거비중(%)": (past_n / past_den * 100).round(3).values,
    })
    out["영역"] = pd.Categorical(out["영역"], AREAS, ordered=True)
    return out.sort_values(["영역", "영역내순위"]).reset_index(drop=True)


AREA_TOP = pd.concat([area_top_keywords(m, CONFIG["AREA_TOP_N"]) for m in MEDIA], ignore_index=True)
for m in MEDIA:
    w = AREA_TOP[(AREA_TOP["매체"] == m) & (AREA_TOP["영역내순위"] <= 10)]
    print(f"── {m}")
    display(w.pivot(index="영역내순위", columns="영역", values="관심대상").reindex(columns=AREAS))

## 9. 매체별 HOT·RISING 선정 *(기준은 잠정)*

| 구분 | 정의 | 사용 지표 |
|---|---|---|
| **HOT** | 현재 구간에서 관심 규모가 큰 대상 | 등장 건수·비중, 출처 다양성(언론사·작성자·채널 수), 월별 지속성(언급이 있었던 달의 비율) |
| **RISING** | 과거(2023~2025년 같은 기간 합산)보다 2026년 등장 *비중*이 뚜렷이 늘어난 대상 | 비중 변화(%p), 증가 배율, 증가 건수, 월별 지속성, 출처 다양성 |
| **신규 포착** | 과거 구간 0건 → 2026년 최소 건수 이상 | 증가율 무한대로 순위를 매기지 않고 따로 분리 |

- 현재 구간 = 2026-01-01~10-26, 과거 구간 = 2023·2024·2025년의 01-01~10-26을 합친 것(건수·분모를 각각 더해 비중 계산). 전년(2025) 동기간 비중도 참고로 함께 적습니다.
- **의존성 점검.** 한 달에 언급이 몰렸거나(`일시 급등 의존`) 한 작성자·채널·언론사가 대부분을 만들었으면(`반복 생산 의존`) 목록에서 *보류*하고 사유를 남깁니다.
- **민감도.** 2023년 SNS 1~10월은 수집 방식이 다른 보완 수집분이므로, 과거 구간을 2024~2025년만으로 바꿔도 RISING 조건(증가 배율)을 유지하는지 `민감도_유지` 열에 표시합니다. 인스타는 여기에 더해 9~10월 수집 편중을 빼고 1~8월끼리 비교합니다.
- **영역별 HOT·RISING(9.2).** 위에서 '선정' 판정을 받은 대상을 6개 영역으로 나눠 매체별·영역별 상위 `AREA_HOT_K`·`AREA_RISING_K`개를 뽑습니다(복합명사 중심 기준은 8.1과 같음).

**SNS 반응은 보조 근거입니다.** 현재 누적 반응으로 과거 대비 반응이 늘었다고 단정하지 않습니다.

In [ ]:
def period_stats(media: str, start, end=None) -> pd.DataFrame:
    """기간 하나(또는 여러 기간을 합친 것)의 관심대상별 지표를 계산합니다.

    start에 [(시작, 종료), …] 목록을 주면 그 기간들을 합쳐 계산합니다(건수·분모·달 수를 각각 더함).

    Returns (target 인덱스)
    -------
    n          : 등장 건수
    share      : 등장 비중
    diversity  : 출처 다양성(언론사·작성자·채널 수)
    active     : 언급이 있었던 달 수
    months     : 구간의 달 수
    top_src    : 가장 많이 만든 출처 하나의 비중(반복 생산 점검)
    spike      : 가장 많은 달 하나의 비중(일시 급등 점검)
    """
    periods = start if isinstance(start, list) else [(start, end)]
    c = pd.concat([subset_ct(media, a, b, trend=True) for a, b in periods])
    n_den = sum(denominator(media, a, b, trend=True) for a, b in periods)
    months = sum(len(pd.period_range(a, b, freq="M")) for a, b in periods)
    g = c.groupby("target")
    out = pd.DataFrame({
        "n": g["count_key"].nunique(),
        "diversity": g["source"].nunique(),
        "active": g["month"].nunique(),
    })
    out["share"] = out["n"] / n_den
    out["months"] = months
    src = c.groupby(["target", "source"])["count_key"].nunique()
    out["top_src"] = src.groupby(level=0).max() / out["n"]
    out["top3_src"] = src.groupby(level=0).apply(lambda x: x.nlargest(3).sum()) / out["n"]
    mon = c.groupby(["target", "month"])["count_key"].nunique()
    out["spike"] = mon.groupby(level=0).max() / out["n"]
    out.attrs["denominator"] = n_den
    return out


def select_hot(media: str) -> pd.DataFrame:
    """현재 구간(CUR_PERIOD) HOT 후보표를 만들고 기준 충족 여부와 보류 사유를 표시합니다."""
    s = period_stats(media, *CONFIG["CUR_PERIOD"])
    s = s[s["n"] >= CONFIG["HOT_MIN_COUNT"][media] * 0.5]          # 표 크기 관리: 아주 작은 대상 제외
    s["active_ratio"] = s["active"] / s["months"]
    reasons = pd.Series("", index=s.index)
    reasons[s["n"] < CONFIG["HOT_MIN_COUNT"][media]] += "최소건수 미달;"
    reasons[s["diversity"] < CONFIG["HOT_MIN_DIVERSITY"][media]] += "출처다양성 미달;"
    reasons[s["active_ratio"] < CONFIG["HOT_MIN_ACTIVE_RATIO"]] += "지속성 미달;"
    reasons[s["top_src"] > CONFIG["TOP_SOURCE_MAX_SHARE"]] += "반복 생산 의존;"
    reasons[s["top3_src"] > CONFIG["TOP3_SOURCE_MAX_SHARE"]] += "소수 출처 집중;"
    s["보류사유"] = reasons
    s["판정"] = np.where(reasons == "", "선정", "보류")
    s = s.sort_values(["판정", "n"], ascending=[False, False])
    sel = s[s["판정"] == "선정"].head(CONFIG["HOT_TOP_K"]).index
    s["HOT"] = s.index.isin(sel)
    s.insert(0, "매체", media)
    return s


def past_windows(years, mmdd_start=None, mmdd_end=None) -> list:
    """과거 연도들의 '현재 구간과 같은 달력 기간' 목록을 만듭니다. 예: [2023, 2024] → [(2023-01-01, 2023-10-26), …]"""
    s0 = mmdd_start or CONFIG["CUR_PERIOD"][0][5:]
    e0 = mmdd_end or CONFIG["CUR_PERIOD"][1][5:]
    return [(f"{y}-{s0}", f"{y}-{e0}") for y in years]


def select_rising(media: str) -> pd.DataFrame:
    """과거(2023~2025년 같은 기간 합산) 대비 2026년 RISING·신규 포착 후보표를 만듭니다.

    비교는 '등장 비중'으로 합니다(수집량 변화 보정). n_prev·share_prev는 과거 구간 합산값이고,
    share_2025는 전년 동기간 비중(참고)입니다. 비중 보정만으로 수집 편향이 사라진다고 보지 않으므로
    민감도 기준(2024~2025년만, 인스타는 1~8월끼리)에서도 증가 배율 조건을 만족하는지 함께 표시합니다.
    """
    cur = period_stats(media, *CONFIG["CUR_PERIOD"])
    prev = period_stats(media, past_windows(CONFIG["PAST_YEARS"]))
    s = cur.join(prev[["n", "share", "diversity"]], rsuffix="_prev", how="left").fillna({"n_prev": 0, "share_prev": 0, "diversity_prev": 0})
    s["share_2025"] = period_stats(media, *CONFIG["PREV_PERIOD"])["share"].reindex(s.index).fillna(0)
    s["증가건수"] = s["n"] - s["n_prev"] / len(CONFIG["PAST_YEARS"])          # 과거 연평균 대비 증가 건수
    s["비중변화_pp"] = (s["share"] - s["share_prev"]) * 100
    s["증가배율"] = np.where(s["share_prev"] > 0, s["share"] / s["share_prev"].replace(0, np.nan), np.inf)

    # 민감도: 과거 = 2024~2025년만(2023 보완 수집분 제외), 인스타는 1~8월끼리
    cy = int(CONFIG["CUR_PERIOD"][0][:4])
    if media == "인스타":
        sm, em = CONFIG["SENS_PERIOD_MMDD"]
        sc = period_stats(media, past_windows([cy], sm, em))
        sp = period_stats(media, past_windows(CONFIG["SENS_PAST_YEARS"], sm, em))
    else:
        sc = cur
        sp = period_stats(media, past_windows(CONFIG["SENS_PAST_YEARS"]))
    sens = sc[["share"]].join(sp[["share"]], rsuffix="_p", how="left").fillna(0)
    sens_ok = (sens["share"] > sens["share_p"] * CONFIG["RISING_MIN_LIFT"]) & (sens["share"] > 0)
    s["민감도_유지"] = s.index.map(lambda t: bool(sens_ok.get(t, False)))

    reasons = pd.Series("", index=s.index)
    reasons[s["n"] < CONFIG["RISING_MIN_COUNT"][media]] += "최소건수 미달;"
    reasons[s["증가건수"] < CONFIG["RISING_MIN_DELTA"][media]] += "증가건수 미달;"
    reasons[(s["증가배율"] < CONFIG["RISING_MIN_LIFT"])] += "증가배율 미달;"
    reasons[s["active"] < CONFIG["RISING_MIN_ACTIVE_MONTHS"]] += "지속성 미달;"
    reasons[s["diversity"] < CONFIG["RISING_MIN_DIVERSITY"][media]] += "출처다양성 미달;"
    reasons[s["spike"] > CONFIG["SPIKE_MAX_SHARE"]] += "일시 급등 의존;"
    reasons[s["top_src"] > CONFIG["TOP_SOURCE_MAX_SHARE"]] += "반복 생산 의존;"
    reasons[s["top3_src"] > CONFIG["TOP3_SOURCE_MAX_SHARE"]] += "소수 출처 집중;"
    s["보류사유"] = reasons
    s["구분"] = np.where(s["n_prev"] == 0, "신규 포착", "RISING")
    s["판정"] = np.where(reasons == "", "선정", "보류")
    s = s[s["n"] >= CONFIG["RISING_MIN_COUNT"][media] * 0.5]     # 표 크기 관리를 위해 아주 작은 대상은 제외
    s = s.sort_values(["판정", "비중변화_pp"], ascending=[False, False])
    top = s[(s["판정"] == "선정") & (s["구분"] == "RISING")].head(CONFIG["RISING_TOP_K"]).index
    new = s[(s["판정"] == "선정") & (s["구분"] == "신규 포착")].head(CONFIG["RISING_TOP_K"]).index
    s["RISING"] = s.index.isin(top)
    s["신규포착"] = s.index.isin(new)
    s.insert(0, "매체", media)
    return s


HOT = {m: select_hot(m) for m in MEDIA}
RISING = {m: select_rising(m) for m in MEDIA}
for m in MEDIA:
    print(f"{m}: HOT {HOT[m]['HOT'].sum()}개, RISING {RISING[m]['RISING'].sum()}개, 신규 포착 {RISING[m]['신규포착'].sum()}개")

In [ ]:
def change_pattern(media: str, targets: list) -> pd.DataFrame:
    """공통 관측기간의 월별 등장 건수로 변화 유형을 붙입니다(규칙은 잠정).

    - 일시 급등형: 가장 많은 한 달이 전체의 40% 이상
    - 계절형: 같은 달력월(앞뒤 1개월 포함)에 2개 이상 연도에서 피크가 나타나고, 그 3개월에 전체의 50% 이상이 몰림
    - 지속형: 언급이 있었던 달이 전체 달의 70% 이상
    - 간헐형: 그 밖의 경우
    """
    c = subset_ct(media, CONFIG["COMMON_START"], CONFIG["COMMON_END"], trend=True)
    c = c[c["target"].isin(targets)]
    mon = c.groupby(["target", "month"])["count_key"].nunique()
    n_months = len(pd.period_range(CONFIG["COMMON_START"], CONFIG["COMMON_END"], freq="M"))
    rows = []
    for t in targets:
        if t not in mon.index.get_level_values(0):
            rows.append((t, "자료 부족", 0, 0)); continue
        s = mon.loc[t]
        total = s.sum()
        per = pd.PeriodIndex(s.index, freq="M")
        peak_share = s.max() / total
        active_ratio = len(s) / n_months
        # 계절성: 연도별 피크 달력월
        df_ = pd.DataFrame({"y": per.year, "m": per.month, "n": s.values})
        peaks = df_.loc[df_.groupby("y")["n"].idxmax(), "m"]
        cal = df_.groupby("m")["n"].sum().reindex(range(1, 13), fill_value=0)
        top_m = int(cal.idxmax())
        window = [((top_m - 2) % 12) + 1, top_m, (top_m % 12) + 1]
        seasonal = (peaks.isin(window).sum() >= 2) and (cal.loc[window].sum() / total >= 0.5)
        if peak_share >= 0.4:
            p = "일시 급등형"
        elif seasonal:
            p = f"계절형({top_m}월 전후)"
        elif active_ratio >= 0.7:
            p = "지속형"
        else:
            p = "간헐형"
        rows.append((t, p, round(active_ratio, 2), round(peak_share, 2)))
    return pd.DataFrame(rows, columns=["target", "변화유형", "활동월비율", "최대월비중"]).set_index("target")


for m in MEDIA:
    tg = list(set(HOT[m].index[HOT[m]["HOT"]]) | set(RISING[m].index[RISING[m]["RISING"] | RISING[m]["신규포착"]]))
    pat = change_pattern(m, tg)
    HOT[m] = HOT[m].join(pat, how="left")
    RISING[m] = RISING[m].join(pat, how="left")
    for tbl in (HOT[m], RISING[m]):
        tbl["영역"] = tbl.index.map(lambda t: AREA_MAP.get(t) or classify_area(t)[0])

_cols_h = ["매체", "n", "share", "diversity", "active", "top_src", "영역", "변화유형"]
display(pd.concat({m: HOT[m][HOT[m]["HOT"]].head(15)[["n", "diversity", "영역", "변화유형"]].reset_index() for m in MEDIA}, axis=1))

In [ ]:
_cols_r = ["n_prev", "n", "비중변화_pp", "증가배율", "민감도_유지", "영역", "변화유형"]
for m in MEDIA:
    print(f"── {m} RISING")
    display(RISING[m][RISING[m]["RISING"]].head(15)[_cols_r])
    print(f"── {m} 신규 포착")
    display(RISING[m][RISING[m]["신규포착"]].head(10)[["n", "diversity", "spike", "영역", "변화유형"]])

### 9.2 영역별 HOT·RISING (매체별)

9단계에서 *선정* 판정을 받은 대상 전체를 6개 영역으로 나누고, 매체별·영역별로 HOT은 2026년 등장 건수 순, RISING은 비중 변화(%p) 순으로 뽑습니다. RISING 표에는 과거 0건이었던 **신규 포착**도 함께 넣고 `구분` 열로 표시합니다. 기준을 충족한 대상이 적은 영역은 억지로 채우지 않습니다.

In [ ]:
def area_hot_rising(media: str) -> tuple:
    """매체 하나의 영역별 HOT·RISING 표를 만듭니다(선정 판정 대상만, 복합명사 중심).

    Returns
    -------
    hot : DataFrame[매체, 영역, 영역내순위, 관심대상, 2026건수, 2026비중(%), 출처다양성, 활동월, 변화유형]
    ris : DataFrame[매체, 영역, 영역내순위, 관심대상, 구분, 과거건수(2023~25 합산), 과거연평균건수, 2026건수, 과거비중(%), 2026비중(%), 전년비중(%),
                    비중변화(%p), 증가배율, 민감도_유지, 변화유형]
    """
    area = lambda t: AREA_MAP.get(t) or classify_area(t)[0]
    h = HOT[media][HOT[media]["판정"] == "선정"].copy()
    h["영역"] = h.index.map(area)
    h = h[h["영역"].isin(AREAS) & h.index.map(is_specific)].sort_values("n", ascending=False)
    h = h.groupby("영역", sort=False).head(CONFIG["AREA_HOT_K"])
    r = RISING[media][RISING[media]["판정"] == "선정"].copy()
    r["영역"] = r.index.map(area)
    r = r[r["영역"].isin(AREAS) & r.index.map(is_specific)].sort_values("비중변화_pp", ascending=False)
    r = r.groupby("영역", sort=False).head(CONFIG["AREA_RISING_K"])
    pat = change_pattern(media, list(set(h.index) | set(r.index)))["변화유형"]
    hot = pd.DataFrame({"매체": media, "영역": h["영역"], "관심대상": h.index, "2026건수": h["n"],
                        "2026비중(%)": (h["share"] * 100).round(3), "출처다양성": h["diversity"],
                        "활동월": h["active"].astype(int).astype(str) + "/" + h["months"].astype(int).astype(str),
                        "변화유형": pat.reindex(h.index).values})
    ris = pd.DataFrame({"매체": media, "영역": r["영역"], "관심대상": r.index, "구분": r["구분"],
                        "과거건수": r["n_prev"].astype(int),
                        "과거연평균건수": (r["n_prev"] / len(CONFIG["PAST_YEARS"])).round(1), "2026건수": r["n"],
                        "과거비중(%)": (r["share_prev"] * 100).round(3), "2026비중(%)": (r["share"] * 100).round(3),
                        "전년비중(%)": (r["share_2025"] * 100).round(3), "비중변화(%p)": r["비중변화_pp"].round(3),
                        "증가배율": r["증가배율"].replace(np.inf, np.nan).round(2), "민감도_유지": r["민감도_유지"],
                        "변화유형": pat.reindex(r.index).values})
    for t in (hot, ris):
        t["영역"] = pd.Categorical(t["영역"], AREAS, ordered=True)
        t.sort_values(["영역"], kind="stable", inplace=True)
        t.insert(2, "영역내순위", t.groupby("영역", observed=True).cumcount() + 1)
        t.reset_index(drop=True, inplace=True)
    return hot, ris


_ahr = {m: area_hot_rising(m) for m in MEDIA}
AREA_HOT = pd.concat([v[0] for v in _ahr.values()], ignore_index=True)
AREA_RISING = pd.concat([v[1] for v in _ahr.values()], ignore_index=True)
print("영역별 HOT 수:"); display(AREA_HOT.pivot_table(index="영역", columns="매체", values="관심대상", aggfunc="count", observed=False))
print("영역별 RISING 수:"); display(AREA_RISING.pivot_table(index="영역", columns="매체", values="관심대상", aggfunc="count", observed=False))

### 9.1 HOT·RISING 월별 추이

매체별 HOT 상위 5개와 RISING 상위 5개(HOT과 겹치면 다음 순위)의 **월별 등장 비중**(그 달 유효 콘텐츠 대비 %)을 그립니다.
- 회색 음영 = 과거 구간(2023·2024·2025년 1~10월), 파란 음영 = 현재 구간(2026-01~10). RISING 비교에 쓴 구간입니다.
- 인스타그램 2026년 9~10월은 수집 편중 구간이므로 점선으로 표시합니다.
- 유튜브는 연도 비교와 같게 '기본' 수집분만 씁니다.

In [ ]:
def pick_hot_rising(media: str, k: int = 5) -> pd.DataFrame:
    """그래프에 그릴 대상을 고릅니다: HOT 상위 k개 + RISING 상위 k개(HOT과 겹치는 대상은 건너뜀)."""
    hot = HOT[media].index[HOT[media]["HOT"]][:k].tolist()
    ris = [t for t in RISING[media].index[RISING[media]["RISING"]] if t not in hot][:k]
    return pd.DataFrame({"target": hot + ris, "구분": ["HOT"] * len(hot) + ["RISING"] * len(ris)})


def hot_rising_monthly(media: str, k: int = 5) -> pd.DataFrame:
    """HOT·RISING 대상의 월별 등장 건수·비중을 긴 표(long table)로 반환합니다.

    Returns
    -------
    DataFrame[매체, 구분, 순서, 관심대상, 월, 등장건수, 분모, 등장비중(%)]
    분모 = 그 달 유효 콘텐츠 수(유튜브는 기본 수집분). 언급이 없는 달은 0으로 채웁니다.
    공통 관측기간이 월 중간에 시작하면 그 달은 분모가 너무 작아 빼고, 첫 온전한 달부터 그립니다.
    """
    sel = pick_hot_rising(media, k)
    share = monthly_series(media, sel["target"].tolist(), trend=True)          # 월 × 대상 비중(0~1)
    start = pd.Timestamp(CONFIG["COMMON_START"])                               # 시작 월이 온전하지 않으면 제외
    first_full = (start if start.day == 1 else start + pd.offsets.MonthBegin(1)).strftime("%Y-%m")
    share = share[share.index >= first_full]
    c = subset_ct(media, CONFIG["COMMON_START"], CONFIG["COMMON_END"], trend=True)
    cnt = (c[c["target"].isin(sel["target"])].groupby(["month", "target"])["count_key"].nunique()
           .unstack(fill_value=0).reindex(index=share.index, columns=share.columns, fill_value=0))
    b = BASE[media]
    b = b[in_period(b["date"], CONFIG["COMMON_START"], CONFIG["COMMON_END"])]
    if media == "유튜브" and CONFIG["YT_TREND_BASIS_ONLY"]:
        b = b[b["collect_mode"] == "기본"]
    den = b.groupby("month")["count_key"].nunique().reindex(share.index)
    rows = []
    for order, (_, r) in enumerate(sel.iterrows()):
        t = r["target"]
        for mon in share.index:
            rows.append((media, r["구분"], order, t, mon, int(cnt.at[mon, t]) if t in cnt else 0,
                         int(den.get(mon, 0)), round(float(share.at[mon, t]) * 100, 3) if t in share else 0.0))
    return pd.DataFrame(rows, columns=["매체", "구분", "순서", "관심대상", "월", "등장건수", "분모", "등장비중(%)"])


def plot_hot_rising(media: str, df: pd.DataFrame):
    """HOT 5개(윗줄)와 RISING 5개(아랫줄)의 월별 등장 비중을 작은 그래프 10개로 그립니다."""
    fig, axes = plt.subplots(2, 5, figsize=(16, 5.5), sharex=True)
    months = sorted(df["월"].unique())
    x = np.arange(len(months))
    cur0, cur1 = months.index(CONFIG["CUR_PERIOD"][0][:7]), months.index(CONFIG["CUR_PERIOD"][1][:7])
    past = [(months.index(a[:7]), months.index(b[:7])) for a, b in past_windows(CONFIG["PAST_YEARS"]) if a[:7] in months]
    for ax in axes.flat:
        ax.set_visible(False)
    for grp, row in (("HOT", 0), ("RISING", 1)):
        tg = df[df["구분"] == grp].drop_duplicates("관심대상")["관심대상"].tolist()
        for j, t in enumerate(tg[:5]):
            ax = axes[row, j]
            ax.set_visible(True)
            y = df[df["관심대상"] == t].set_index("월").reindex(months)["등장비중(%)"].values
            for p0, p1 in past:
                ax.axvspan(p0 - 0.5, p1 + 0.5, color="0.92")
            ax.axvspan(cur0 - 0.5, cur1 + 0.5, color="#dbe7f5")
            if media == "인스타":
                cut = months.index("2026-09")
                ax.plot(x[:cut + 1], y[:cut + 1], color="#2a6fb0", lw=1.4)
                ax.plot(x[cut:], y[cut:], color="#2a6fb0", lw=1.4, ls="--")
            else:
                ax.plot(x, y, color="#2a6fb0", lw=1.4)
            ax.set_title(f"[{grp}] {t}", fontsize=10)
            ax.set_ylim(bottom=0)
            ax.tick_params(labelsize=8)
    ticks = [i for i, m in enumerate(months) if m.endswith("-01")]
    for ax in axes[1]:
        ax.set_xticks(ticks)
        ax.set_xticklabels([months[i][:4] for i in ticks])
    fig.suptitle(f"{media}: HOT·RISING 대상 월별 등장 비중(%) — 회색 과거(2023~2025 1~10월), 파랑 2026-01~10", fontsize=12)
    plt.tight_layout()
    plt.show()


HR_MONTHLY = pd.concat([hot_rising_monthly(m) for m in MEDIA], ignore_index=True)
for m in MEDIA:
    plot_hot_rising(m, HR_MONTHLY[HR_MONTHLY["매체"] == m])

## 10. SNS 반응 기반 관심도 분석

- 인스타와 유튜브의 원시 수치는 **합산하지 않습니다**. 조회수는 노출·소비 규모, 좋아요·댓글은 이용자 반응 규모로 구분합니다.
- 좋아요 숨김, 조회수 미제공, 수집 실패는 **0이 아니라 결측**입니다. 지표마다 유효 관측 수를 함께 표시합니다.
- 오래된 콘텐츠일수록 누적 반응에 유리하므로 게시 후 경과기간별로 비교합니다. 반복 수집 이력이 없어 *반응 증가의 시계열*은 분석할 수 없습니다.
- 한 콘텐츠에 여러 대상이 나오면 조회수를 대상별로 나누지 않고 **'해당 대상을 포함한 콘텐츠의 반응'**으로 표기합니다.
- 댓글이 많다는 것을 긍정적 관심, 방문·구매·호감도와 같은 것으로 보지 않습니다.

In [ ]:
SNS = {"인스타": BASE["인스타"].copy(), "유튜브": BASE["유튜브"].copy()}
METRICS = ["views", "likes", "comments"]
METRIC_KO = {"views": "조회수", "likes": "좋아요", "comments": "댓글수"}


def reaction_quality() -> pd.DataFrame:
    """플랫폼×콘텐츠유형별 지표 유효 관측 수와 결측률."""
    rows = []
    for p, d in SNS.items():
        for ty, g in d.groupby("type"):
            r = {"플랫폼": p, "유형": ty, "콘텐츠수": len(g)}
            for mt in METRICS:
                r[f"{METRIC_KO[mt]}_유효"] = int(g[mt].notna().sum())
                r[f"{METRIC_KO[mt]}_결측률"] = round(g[mt].isna().mean(), 3)
            if p == "인스타":
                r["좋아요숨김"] = int(g["likes_hidden"].sum())
            rows.append(r)
    return pd.DataFrame(rows)


def reaction_distribution() -> pd.DataFrame:
    """플랫폼×유형×게시연도별 지표 분포(유효 관측 수, 중앙값, 75·90분위, 최대값)."""
    rows = []
    for p, d in SNS.items():
        for (ty, y), g in d.groupby(["type", "year"]):
            for mt in METRICS:
                s = g[mt].dropna()
                if len(s) == 0:
                    continue
                rows.append({"플랫폼": p, "유형": ty, "게시연도": y, "지표": METRIC_KO[mt], "유효관측": len(s),
                             "중앙값": s.median(), "p75": s.quantile(.75), "p90": s.quantile(.9), "최대": s.max()})
    return pd.DataFrame(rows)


def add_rates_and_age(d: pd.DataFrame) -> pd.DataFrame:
    """좋아요율·댓글률(조회수가 MIN_VIEWS_FOR_RATE 이상일 때만), 게시 후 경과일과 경과 구간을 붙입니다."""
    d = d.copy()
    ok = d["views"] >= CONFIG["MIN_VIEWS_FOR_RATE"]
    d["좋아요율"] = np.where(ok & d["likes"].notna(), d["likes"] / d["views"], np.nan)
    d["댓글률"] = np.where(ok & d["comments"].notna(), d["comments"] / d["views"], np.nan)
    d["경과일"] = (pd.Timestamp(CONFIG["COLLECTION_DATE"]) - d["date"]).dt.days
    d["경과구간"] = pd.cut(d["경과일"], [-1, 30, 90, 180, 365, 730, 10_000],
                         labels=["30일 이내", "31~90일", "91~180일", "181~365일", "1~2년", "2년 초과"])
    return d


for p in SNS:
    SNS[p] = add_rates_and_age(SNS[p])

REACT_QC = reaction_quality()
REACT_DIST = reaction_distribution()
display(REACT_QC)

In [ ]:
def age_effect() -> pd.DataFrame:
    """경과 구간별 반응 중앙값. 오래된 콘텐츠의 누적 우위가 얼마나 큰지 보여 줍니다.

    주의: 같은 콘텐츠를 여러 번 수집한 이력이 없으므로, 이 표를 '반응 속도'로 해석하지 않습니다.
    """
    rows = []
    for p, d in SNS.items():
        for (ty, ag), g in d.groupby(["type", "경과구간"], observed=True):
            r = {"플랫폼": p, "유형": ty, "경과구간": ag, "콘텐츠수": len(g)}
            for mt in METRICS:
                r[f"{METRIC_KO[mt]}_중앙값"] = g[mt].median()
            rows.append(r)
    return pd.DataFrame(rows)


AGE_EFFECT = age_effect()
AGE_EFFECT

In [ ]:
def relative_index(d: pd.DataFrame, metric: str) -> pd.Series:
    """상대 반응 지수 = 콘텐츠 지표 ÷ (같은 플랫폼·유형·게시 분기 콘텐츠의 중앙값).

    유형(숏츠/일반, 릴스/사진/캐러셀)과 게시 시기에 따른 누적 반응 차이를 줄여 대상 간 비교에 씁니다.
    분모 중앙값이 0이거나 지표가 결측이면 NaN입니다.
    """
    q = d["date"].dt.to_period("Q").astype(str)
    med = d.groupby([d["type"], q])[metric].transform("median")
    return d[metric] / med.replace(0, np.nan)


def high_reaction_list() -> pd.DataFrame:
    """플랫폼×유형×지표별 상위 HIGH_REACTION_TOP_N개 고반응 콘텐츠와 그 안의 관심대상(산출물 ⑤)."""
    out = []
    for p, d in SNS.items():
        tg = CT[p].groupby("content_id")["target"].apply(lambda s: ", ".join(s.head(10)))
        for ty, g in d.groupby("type"):
            for mt in METRICS:
                s = g.dropna(subset=[mt]).nlargest(CONFIG["HIGH_REACTION_TOP_N"], mt)
                if s.empty:
                    continue
                t = pd.DataFrame({
                    "플랫폼": p, "유형": ty, "기준지표": METRIC_KO[mt], "게시일": s["date"].dt.strftime("%Y-%m-%d"),
                    "ID": s["content_id"], "제목·본문": (s["title"].where(s["title"] != "", s["body"])).str[:100].str.replace("\n", " "),
                    "출처": s["source"], "조회수": s["views"], "좋아요": s["likes"], "댓글수": s["comments"],
                    "관련 관심대상": s["content_id"].map(tg).fillna(""), "URL": s["url"],
                    "결측지표": s[METRICS].isna().apply(lambda r: ",".join(METRIC_KO[k] for k, v in r.items() if v), axis=1),
                    "경과구간": s["경과구간"].astype(str),
                })
                out.append(t)
    return pd.concat(out, ignore_index=True)


for p in SNS:
    for mt in METRICS:
        SNS[p][f"rel_{mt}"] = relative_index(SNS[p], mt)

HIGH_REACT = high_reaction_list()
HIGH_REACT[HIGH_REACT["기준지표"] == "조회수"].groupby(["플랫폼", "유형"]).head(3)

In [ ]:
def target_reaction(platform: str, targets=None) -> pd.DataFrame:
    """관심대상별 반응 요약('해당 대상을 포함한 콘텐츠의 반응').

    지표별 유효 관측 수, 합계, 중앙값, 최대값, 상위 1개 콘텐츠 집중도(최대 ÷ 합계), 상대 반응 지수 중앙값을 계산합니다.
    대상별 합계는 서로 겹칠 수 있으므로(한 콘텐츠에 여러 대상) 더하지 않습니다.
    """
    c = CT[platform][["content_id", "target"]]
    if targets is not None:
        c = c[c["target"].isin(targets)]
    d = SNS[platform][["content_id"] + METRICS + [f"rel_{m}" for m in METRICS]]
    x = c.merge(d, on="content_id")
    g = x.groupby("target")
    out = pd.DataFrame({"콘텐츠수": g["content_id"].nunique()})
    for mt in METRICS:
        k = METRIC_KO[mt]
        out[f"{k}_유효"] = g[mt].count()
        out[f"{k}_합계"] = g[mt].sum(min_count=1)
        out[f"{k}_중앙값"] = g[mt].median()
        out[f"{k}_최대"] = g[mt].max()
        out[f"{k}_상위1집중도"] = out[f"{k}_최대"] / out[f"{k}_합계"]
        out[f"{k}_상대지수중앙값"] = g[f"rel_{mt}"].median()
    return out


def mention_vs_reaction(platform: str, top_n=300) -> pd.DataFrame:
    """언급 규모와 반응 수준을 나눠 봅니다.

    - 언급多·반응低: 등장 건수 상위 25% 안, 상대 반응 지수 중앙값 < 1
    - 언급少·반응高: 등장 건수 하위 50%, 상대 반응 지수 중앙값 ≥ 2
    반응 지수는 인스타=좋아요, 유튜브=조회수 기준입니다(지표가 가장 많이 관측됨).
    """
    key = "좋아요" if platform == "인스타" else "조회수"
    tg = FREQ_COMMON[platform].head(top_n)["target"].tolist()
    r = target_reaction(platform, tg)
    q75, q50 = r["콘텐츠수"].quantile(.75), r["콘텐츠수"].quantile(.5)
    idx = r[f"{key}_상대지수중앙값"]
    r["구분"] = np.select([(r["콘텐츠수"] >= q75) & (idx < 1), (r["콘텐츠수"] <= q50) & (idx >= 2)],
                        ["언급多·반응低", "언급少·반응高"], "")
    r.insert(0, "플랫폼", platform)
    r["영역"] = r.index.map(lambda t: AREA_MAP.get(t, ""))
    return r.sort_values("콘텐츠수", ascending=False)


MVR = {p: mention_vs_reaction(p) for p in SNS}
for p in SNS:
    print(f"── {p}")
    display(MVR[p][MVR[p]["구분"] != ""].groupby("구분").head(8)[["콘텐츠수", "구분", "영역"] + [c for c in MVR[p].columns if "상대지수" in c][:2]])

## 11. 콘텐츠 생산 주체 보조 분류

`dict/생산주체규칙.csv`의 규칙으로 **언론·방송 보도 / 지자체·공공기관 공식 홍보 / 기업·점포 홍보 및 협찬 / 일반 이용자·크리에이터 / 판별 불가**로 나눕니다.

- 유튜브는 채널명, 인스타는 작성자가 비식별 처리되어 있으므로 **본문 속 협찬·광고 표기 문구**만 근거로 씁니다.
- 계정명만으로 광고·협찬을 단정하지 않으며, 행마다 `판별근거`를 남깁니다.
- 근거가 없으면 유튜브는 '일반 이용자·크리에이터(추정)', 인스타는 '판별 불가'로 둡니다. 인스타에서 게시물이 `REPEAT_AUTHOR_MIN_POSTS`개 이상인 작성자는 `반복게시계정`으로 따로 표시해, 공식 홍보처럼 반복 생산된 글과 여러 이용자의 자발적 언급을 나눠 해석할 수 있게 합니다.

In [ ]:
PROD_RULES = pd.read_csv(DICT_DIR / "생산주체규칙.csv").sort_values("우선순위")
PROD_RULES["re"] = PROD_RULES["패턴"].map(lambda p: re.compile(p, re.I))


def classify_producer(platform: str, row) -> tuple:
    """콘텐츠 하나의 생산 주체와 판별 근거를 돌려줍니다. 규칙은 우선순위 순서로 적용합니다."""
    for _, r in PROD_RULES.iterrows():
        if r["대상필드"] == "채널명" and platform != "유튜브":
            continue
        field = row["source"] if r["대상필드"] == "채널명" else f"{row['title']} {row['body']}"
        m = r["re"].search(str(field))
        if m:
            return r["주체"], f"{r['대상필드']}에 '{m.group(0)}'"
    if platform == "유튜브":
        return "일반 이용자·크리에이터(추정)", "기관·언론·업체 표기 없음"
    return "판별 불가", "작성자 비식별, 본문에 협찬 표기 없음"


for p in SNS:
    res = [classify_producer(p, r) for _, r in SNS[p][["source", "title", "body"]].iterrows()]
    SNS[p]["생산주체"] = [r[0] for r in res]
    SNS[p]["판별근거"] = [r[1] for r in res]
cnt = SNS["인스타"]["source"].value_counts()
SNS["인스타"]["반복게시계정"] = SNS["인스타"]["source"].map(cnt) >= CONFIG["REPEAT_AUTHOR_MIN_POSTS"]

PRODUCER_DIST = pd.concat({p: SNS[p]["생산주체"].value_counts() for p in SNS}, axis=1).fillna(0).astype(int)
display(PRODUCER_DIST)
print("인스타 반복 게시 계정 게시물 비중:", f"{SNS['인스타']['반복게시계정'].mean():.1%}")

In [ ]:
def producer_mix(platform: str, targets: list) -> pd.DataFrame:
    """관심대상별 생산 주체 구성비. 공식 홍보·언론 보도가 대부분인 대상을 표시할 수 있습니다."""
    x = CT[platform][CT[platform]["target"].isin(targets)].merge(
        SNS[platform][["content_id", "생산주체"] + (["반복게시계정"] if platform == "인스타" else [])], on="content_id")
    mix = pd.crosstab(x["target"], x["생산주체"], normalize="index").round(3)
    if platform == "인스타":
        mix["반복게시계정_비중"] = x.groupby("target")["반복게시계정"].mean().round(3)
    return mix


PROD_MIX = {}
for p in SNS:
    tg = list(set(HOT[p].index[HOT[p]["HOT"]]) | set(RISING[p].index[RISING[p]["RISING"] | RISING[p]["신규포착"]]))
    PROD_MIX[p] = producer_mix(p, tg)
PROD_MIX["유튜브"].head(10)

## 12. 매체 간 공통·차별 관심대상

- 매체별 상위 `CROSS_TOP_K`개 대상의 출현 여부로 **3매체 공통 / 2매체 공통 / 뉴스 중심 / 인스타 중심 / 유튜브 중심**을 나눕니다.
- 원시 지표를 합산한 통합 순위는 만들지 않습니다. 매체별 순위와 비중을 나란히 놓고 비교합니다.
- **영역 구성비**는 *주분류 기준*이고, 분모는 매체별 상위 대상 가운데 분류된 대상 수입니다.
- **맥락 비교:** 공통 대상과 함께 나온 단어(보조어 포함)를 매체별로 보여 줍니다. 예를 들어 뉴스는 '예산·공항', SNS는 '여행·드라이브'처럼 다르게 다뤄지는지 확인합니다.

In [ ]:
def cross_media_table(k: int) -> pd.DataFrame:
    """매체별 상위 k개 관심대상의 매체 간 출현 구분표."""
    rank = {m: FREQ_COMMON[m].set_index("target") for m in MEDIA}
    tops = {m: set(rank[m].head(k).index) for m in MEDIA}
    allt = sorted(set.union(*tops.values()))
    rows = []
    for t in allt:
        inm = [m for m in MEDIA if t in tops[m]]
        cat = "3매체 공통" if len(inm) == 3 else ("2매체 공통(" + "·".join(inm) + ")" if len(inm) == 2 else f"{inm[0]} 중심")
        r = {"관심대상": t, "구분": cat, "영역": AREA_MAP.get(t) or classify_area(t)[0]}
        for m in MEDIA:
            r[f"{m}_순위"] = rank[m]["순위"].get(t, np.nan)
            r[f"{m}_비중"] = rank[m]["등장비중"].get(t, 0.0)
        rows.append(r)
    return pd.DataFrame(rows).sort_values(["구분", "뉴스_순위"])


def area_composition(k: int) -> pd.DataFrame:
    """매체별 상위 k개 대상의 영역 구성비(주분류 기준, 분모=상위 k개 중 분류된 대상 수)."""
    rows = {}
    for m in MEDIA:
        t = FREQ_COMMON[m].head(k)["target"]
        a = t.map(lambda x: AREA_MAP.get(x) or classify_area(x)[0])
        a = a[a != "미분류(검토필요)"]
        rows[m] = a.value_counts(normalize=True).reindex(AREAS, fill_value=0)
        rows[m]["분모(분류된 대상 수)"] = len(a)
    return pd.DataFrame(rows)


CROSS = cross_media_table(CONFIG["CROSS_TOP_K"])
AREA_COMP = area_composition(CONFIG["CROSS_TOP_K"])
display(CROSS["구분"].value_counts())
display(AREA_COMP.round(3))
CROSS[CROSS["구분"] == "3매체 공통"].head(20)

### 12.1 영역 구성비 월별 추이

매체별로 **달마다 6개 영역이 차지하는 비중**을 누적 막대(100%)로 그립니다.
- **세는 단위:** 콘텐츠×영역. 한 콘텐츠에 같은 영역 대상이 여러 개 나와도 그 영역은 1번만 셉니다(예: 한 게시물에 내장산·내소사가 함께 나오면 장소·공간 1건).
- **분류 대상:** 8단계에서 매체별 상위 `AREA_CLASSIFY_TOP`개 후보에 붙인 주분류를 씁니다. 미분류 대상은 빼고, 분류된 언급 안에서 100%가 되도록 나눕니다. 달마다 *분류 대상이 한 번이라도 나온 콘텐츠 비율*은 `분류포괄률(%)` 열에 따로 적습니다.
- 수기 분류(`dict/영역분류_수기.csv`)를 채우고 다시 실행하면 그래프가 바로 바뀝니다.
- 유튜브는 연도 비교와 같게 '기본' 수집분만 씁니다.
- 인스타그램 2026년 9~10월은 수집 편중 구간이라 막대를 흐리게 표시합니다. 비중 자체는 계산되지만 해석에 주의해야 합니다.

In [ ]:
AREA_COLORS = dict(zip(AREAS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))   # 색각이상 검증을 통과한 6색


def area_monthly(media: str) -> pd.DataFrame:
    """월별 영역 구성비를 긴 표(long table)로 반환합니다.

    Returns
    -------
    DataFrame[매체, 월, 영역, 영역건수, 영역비중(%), 분류언급합계, 유효콘텐츠수, 분류포괄률(%)]
    - 영역건수: 그 달에 그 영역 대상이 나온 콘텐츠 수(콘텐츠×영역 고유 쌍)
    - 영역비중(%): 영역건수 ÷ 그 달 6개 영역 영역건수 합계 × 100 (미분류 제외, 합계 100%)
    - 분류포괄률(%): 분류된 대상이 하나라도 나온 콘텐츠 ÷ 그 달 유효 콘텐츠 × 100
    """
    c = subset_ct(media, CONFIG["COMMON_START"], CONFIG["COMMON_END"], trend=True)
    c = c.assign(영역=c["target"].map(AREA_MAP))
    c = c[c["영역"].isin(AREAS)]
    start = pd.Timestamp(CONFIG["COMMON_START"])
    first_full = (start if start.day == 1 else start + pd.offsets.MonthBegin(1)).strftime("%Y-%m")
    c = c[c["month"] >= first_full]
    cnt = (c.groupby(["month", "영역"])["count_key"].nunique()
           .unstack(fill_value=0).reindex(columns=AREAS, fill_value=0))
    covered = c.groupby("month")["count_key"].nunique()
    b = BASE[media]
    b = b[in_period(b["date"], CONFIG["COMMON_START"], CONFIG["COMMON_END"])]
    if media == "유튜브" and CONFIG["YT_TREND_BASIS_ONLY"]:
        b = b[b["collect_mode"] == "기본"]
    den = b.groupby("month")["count_key"].nunique()
    den = den[den.index >= first_full]
    cnt = cnt.reindex(den.index, fill_value=0)
    total = cnt.sum(axis=1)
    share = cnt.div(total.where(total > 0), axis=0).fillna(0) * 100
    out = (cnt.stack().rename("영역건수").to_frame()
           .join(share.stack().rename("영역비중(%)")).reset_index())
    out.columns = ["월", "영역", "영역건수", "영역비중(%)"]
    out["분류언급합계"] = out["월"].map(total)
    out["유효콘텐츠수"] = out["월"].map(den)
    out["분류포괄률(%)"] = (out["월"].map(covered).fillna(0) / out["유효콘텐츠수"] * 100).round(1)
    out["영역비중(%)"] = out["영역비중(%)"].round(2)
    out.insert(0, "매체", media)
    return out


def plot_area_monthly(df: pd.DataFrame):
    """세 매체의 월별 영역 구성비를 위아래로 쌓은 100% 누적 막대그래프로 그립니다."""
    medias = [m for m in MEDIA if m in set(df["매체"])]
    fig, axes = plt.subplots(len(medias), 1, figsize=(15, 3.6 * len(medias)), sharex=False)
    for ax, m in zip(np.atleast_1d(axes), medias):
        w = df[df["매체"] == m].pivot(index="월", columns="영역", values="영역비중(%)").reindex(columns=AREAS)
        months = w.index.tolist()
        x = np.arange(len(months))
        faded = [m == "인스타" and mon >= "2026-09" for mon in months]
        bottom = np.zeros(len(months))
        for a in AREAS:
            v = w[a].values
            ax.bar(x, v, bottom=bottom, width=0.82, color=AREA_COLORS[a], label=a,
                   edgecolor="white", linewidth=1.0)
            for i in np.where(faded)[0]:                       # 수집 편중 구간 흐리게
                ax.bar(x[i], v[i], bottom=bottom[i], width=0.82, color="white", alpha=0.55, linewidth=0)
            bottom += v
        ticks = [i for i, mon in enumerate(months) if mon.endswith("-01") or mon.endswith("-07")]
        ax.set_xticks(ticks)
        ax.set_xticklabels([months[i] for i in ticks], fontsize=8)
        ax.set_ylim(0, 100)
        ax.set_yticks([0, 25, 50, 75, 100])
        ax.tick_params(axis="y", labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        cov = df[df["매체"] == m].drop_duplicates("월")["분류포괄률(%)"].median()
        ax.set_title(f"{m}: 월별 영역 구성비(%)  · 분류 대상이 나온 콘텐츠 비율(월 중앙값) {cov:.0f}%",
                     fontsize=11, loc="left")
    np.atleast_1d(axes)[0].legend(ncol=6, fontsize=9, loc="lower left", bbox_to_anchor=(0, 1.12), frameon=False)
    plt.tight_layout()
    plt.show()


AREA_MONTHLY = pd.concat([area_monthly(m) for m in MEDIA], ignore_index=True)
plot_area_monthly(AREA_MONTHLY)
AREA_MONTHLY.pivot_table(index=["매체", "영역"], columns=AREA_MONTHLY["월"].str[:4], values="영역건수", aggfunc="sum")

In [ ]:
def context_words(media: str, target: str, k=10) -> str:
    """대상과 같은 콘텐츠에 함께 나온 단어(다른 후보+보조어) 상위 k개를 '단어(건수)' 형식으로 반환합니다."""
    ids = set(CT[media].loc[CT[media]["target"] == target, "content_id"])
    if not ids:
        return ""
    co = pd.concat([CT[media].loc[CT[media]["content_id"].isin(ids), ["content_id", "target"]],
                    AUX[media].loc[AUX[media]["content_id"].isin(ids) & (AUX[media]["judge"] == "보조어"), ["content_id", "target"]]])
    co = co[co["target"] != target].groupby("target")["content_id"].nunique().nlargest(k)
    return ", ".join(f"{w}({n})" for w, n in co.items())


def context_comparison(targets: list) -> pd.DataFrame:
    """공통 대상의 매체별 동시 출현어 비교표."""
    return pd.DataFrame([{"관심대상": t, **{f"{m}_함께나온말": context_words(m, t) for m in MEDIA}} for t in targets])


COMMON_TARGETS = CROSS[CROSS["구분"] == "3매체 공통"]["관심대상"].head(20).tolist()
CONTEXT_CMP = context_comparison(COMMON_TARGETS)
CONTEXT_CMP.head(10)

## 13. 대표 사례 후보와 최종 요약표

**대표 사례(의뢰서 10.2).** 6개 영역별로 상위 대상을 고르고, 매체마다 대표 콘텐츠 1건을 붙여 *후보 풀*을 만듭니다(뉴스는 여러 언론사가 다룬 기사, SNS는 반응이 높은 콘텐츠). 최종 20~30건은 검토자가 원문을 보고 고릅니다. 근거가 약한 영역을 억지로 채우지 않습니다.

**최종 요약표(의뢰서 10.1).** HOT·RISING·신규 포착에 든 대상을 한 행씩 정리합니다.

In [ ]:
def representative_content(media: str, target: str) -> dict:
    """대상의 대표 콘텐츠 1건을 고릅니다. 뉴스=기사군 크기(전재 수)가 큰 기사, SNS=조회수(없으면 좋아요)가 높은 콘텐츠."""
    ids = CT[media].loc[CT[media]["target"] == target, "content_id"]
    if ids.empty:
        return {}
    b = (SNS[media] if media in SNS else BASE[media]).set_index("content_id").loc[ids]
    if media == "뉴스":
        r = b.sort_values(["dup_size", "date"], ascending=[False, False]).iloc[0]
        ident = f"원자료 행 {r['원자료_행번호']} / {r['source']}"
    else:
        sort_col = "views" if b["views"].notna().any() else "likes"
        r = b.sort_values(sort_col, ascending=False, na_position="last").iloc[0]
        ident = r["url"]
    return {"매체": media, "제목·본문": (r["title"] or r["body"])[:120].replace("\n", " "), "발행·게시일": f"{r['date']:%Y-%m-%d}",
            "조회수": r.get("views", np.nan), "좋아요": r.get("likes", np.nan), "댓글수": r.get("comments", np.nan),
            "원문URL·식별정보": ident, "시군": ", ".join(r["sigun_all"])}


def representative_pool(per_area=8) -> pd.DataFrame:
    """영역별 상위 대상(세 매체 상위 대상 중 순위가 가장 좋은 매체 기준)과 대표 콘텐츠 후보 풀."""
    c = CROSS.copy()
    c["최고순위"] = c[[f"{m}_순위" for m in MEDIA]].min(axis=1)
    rows = []
    for area in AREAS:
        sub = c[c["영역"] == area].nsmallest(per_area, "최고순위")
        for _, r in sub.iterrows():
            for m in MEDIA:
                if pd.notna(r[f"{m}_순위"]):
                    rc = representative_content(m, r["관심대상"])
                    if rc:
                        rows.append({"구분(영역)": area, "관심대상": r["관심대상"], "매체간구분": r["구분"],
                                     **rc, "선정 이유(후보)": f"{m} 상위 {int(r[f'{m}_순위'])}위"})
    return pd.DataFrame(rows)


REP_POOL = representative_pool()
print("대표 사례 후보:", len(REP_POOL), "건 / 영역별:", REP_POOL.groupby("구분(영역)")["관심대상"].nunique().to_dict())
REP_POOL.head(10)

In [ ]:
def build_summary() -> pd.DataFrame:
    """최종 요약표: HOT·RISING·신규 포착 대상의 연도별 건수·비중, 변화 유형, 반응, 선정 근거, 대표 사례, 주의사항."""
    rows = []
    for m in MEDIA:
        sel = set(HOT[m].index[HOT[m]["HOT"]]) | set(RISING[m].index[RISING[m]["RISING"] | RISING[m]["신규포착"]])
        yr = YEARLY[m]
        rx = target_reaction(m, list(sel)) if m in SNS else None
        sg = SIGUN_TOP[SIGUN_TOP["매체"] == m]
        for t in sel:
            basis = []
            if t in HOT[m].index and HOT[m].at[t, "HOT"]:
                h = HOT[m].loc[t]
                basis.append(f"HOT: {int(h['n'])}건, 출처 {int(h['diversity'])}곳, 활동 {int(h['active'])}/{int(h['months'])}개월")
            if t in RISING[m].index and (RISING[m].at[t, "RISING"] or RISING[m].at[t, "신규포착"]):
                r = RISING[m].loc[t]
                basis.append(f"{r['구분']}: 과거(2023~25 합산) {int(r['n_prev'])}건→2026 {int(r['n'])}건, 비중 {r['비중변화_pp']:+.2f}%p"
                             + ("" if r["민감도_유지"] else " (민감도 미유지)"))
            notes = []
            if m == "인스타":
                notes.append("인스타 2026 수집 편중(최근 글) — 비중 기준 해석")
            if m == "유튜브":
                notes.append("검색어×연도 상위 50개 수집 구조")
            if m != "뉴스":
                notes.append("2023년 1~10월은 수집 방식이 다른 보완 수집분")
            pat = (HOT[m]["변화유형"].get(t) if t in HOT[m].index else None) or (RISING[m]["변화유형"].get(t) if t in RISING[m].index else "")
            row = {"관심대상": t, "영역": AREA_MAP.get(t) or classify_area(t)[0], "매체": m,
                   "관련 시군(상위)": ", ".join(sg.loc[sg["관심대상"] == t].nsmallest(3, "순위")["시군"]),
                   "변화유형": pat, "HOT·RISING 선정 근거": " / ".join(basis), "해석상 주의사항": "; ".join(notes)}
            for y in (2023, 2024, 2025, 2026):
                row[f"{y}_건수"] = yr[f"{y}_등장건수"].get(t, np.nan) if t in yr.index else np.nan
                row[f"{y}_비중"] = yr[f"{y}_등장비중"].get(t, np.nan) if t in yr.index else np.nan
                row[f"{y}_순위"] = yr[f"{y}_순위"].get(t, np.nan) if t in yr.index else np.nan
            if rx is not None and t in rx.index:
                k = "좋아요" if m == "인스타" else "조회수"
                row["SNS 반응(포함 콘텐츠)"] = f"{k} 중앙값 {rx.at[t, f'{k}_중앙값']:,.0f}, 최대 {rx.at[t, f'{k}_최대']:,.0f} (유효 {int(rx.at[t, f'{k}_유효'])})"
            rc = representative_content(m, t)
            row["대표 사례·출처"] = f"[{rc.get('발행·게시일','')}] {rc.get('제목·본문','')[:60]} | {rc.get('원문URL·식별정보','')}"
            rows.append(row)
    return pd.DataFrame(rows).sort_values(["매체", "HOT·RISING 선정 근거"])


SUMMARY = build_summary()
print("최종 요약표:", SUMMARY.shape)
SUMMARY.head(10)

## 14. 결과 저장

- `output/미지의전북_분석결과_<실행시각>.xlsx`: 모든 결과표(시트별)
- `output/csv_<실행시각>/`: 시트별 UTF-8 BOM CSV
- `output/전처리_추출_분류_기준서_<실행시각>.xlsx`: 적용한 사전, 규칙, 파라미터(산출물 ②)

모든 표의 분모, 단위, 비교기간은 `읽는법` 시트에 정리했습니다.

In [ ]:
def readme_sheet() -> pd.DataFrame:
    """결과 파일의 '읽는법' 시트: 시트별 내용, 분모, 단위, 비교기간, 결측 처리."""
    c = CONFIG
    rows = [
        ("공통", "등장건수=대상이 나온 고유 콘텐츠 수(뉴스=기사군 수). 같은 콘텐츠 안의 반복 등장은 1건"),
        ("공통", "등장비중=등장건수÷같은 매체·기간(·시군)의 유효 콘텐츠 수(분모 열 참조)"),
        ("공통", f"공통 관측기간 {c['COMMON_START']}~{c['COMMON_END']}; RISING 비교 {c['CUR_PERIOD']} vs 과거 {c['PAST_YEARS']}년 같은 기간 합산(민감도: {c['SENS_PAST_YEARS']}년)"),
        ("공통", "SNS 원자료=전북SNS_수집결과_20230101_20261031.xlsx. 2023.1.1~10.30 보완 수집분은 인스타 '인기 게시물(릴스 탭)', 유튜브 '조회수 순 상위 50개'로 수집 방식이 다름"),
        ("공통", "정당명(더불어민주당·민주당·국민의힘 등)은 분석에서 완전히 제외. 제외 건수는 '2_정당명제외' 시트"),
        ("공통", "SNS: 같은 채널·작성자가 콘텐츠 대부분에 반복해 붙인 대상은 출처당 1건만 셈('2_반복생산상쇄' 시트)"),
        ("공통", "SNS: 여행·맛집·카페 일반어(전주여행, 감성카페, 빵지순례 등)는 보조어로 빼고 상호·메뉴 등 구체적 대상만 순위에 넣음"),
        ("공통", "새만금 단독은 지역명으로 빼고 새만금국제공항·새만금이차전지처럼 뒤에 오는 말과 묶은 복합어만 추출"),
        ("공통", f"SNS 지역 관련성 필터: {c['SNS_RELEVANCE']} / 뉴스 원문확인 제외: {c['NEWS_EXCLUDE_REVIEW']}"),
        ("공통", "유튜브 연도 비교·RISING·추세는 '기본' 수집분만 사용(추가=시범 수집분)"),
        ("공통", "SNS 결측(좋아요 숨김, 조회수 미제공, 수집 실패)은 0이 아닌 결측. 지표별 유효 관측 수 표기"),
        ("공통", "뉴스 날짜=검색표시날짜(실제 발행일 확인 불가, 대리 날짜). 분석 텍스트=제목+미리보기"),
        ("공통", "원시 지표를 합산한 통합 순위·통합 관심도 점수는 만들지 않음"),
        ("HOT/RISING", "기준값은 잠정(CONFIG 시트). '보류사유'에 탈락 원인 표기. 신규 포착=과거(2023~2025 같은 기간) 0건. n_prev·share_prev=과거 합산, share_2025=전년 동기간"),
        ("영역별", "3_영역별다빈도·4_영역별HOT·4_영역별RISING: 복합명사·고유명사·인명·특산품·메뉴 중심(단일 일반명사 제외)"),
        ("SNS반응", "반응 지표는 '해당 대상을 포함한 콘텐츠의 반응'. 대상별 합계는 겹칠 수 있어 더하지 않음"),
        ("SNS반응", f"상대지수=콘텐츠 지표÷같은 플랫폼·유형·게시분기 중앙값. 좋아요율·댓글률은 조회수 {c['MIN_VIEWS_FOR_RATE']} 이상만"),
        ("시군", "여러 시군에 걸친 콘텐츠는 각 시군에 1번씩 집계(시군 합계>전체)"),
        ("영역", "규칙 기반 자동분류 + 분석자 1차 수기 검토(dict/영역분류_수기.csv). 매체별 상위 AREA_CLASSIFY_TOP개 후보를 분류"),
    ]
    return pd.DataFrame(rows, columns=["범위", "설명"])


def _flatten(df: pd.DataFrame) -> pd.DataFrame:
    """엑셀 저장용 정리: 목록형 열을 문자열로 바꾸고 인덱스를 열로 꺼냅니다."""
    d = df.copy()
    if not isinstance(d.index, pd.RangeIndex):
        d = d.reset_index()
    for c in d.columns:
        if d[c].map(lambda v: isinstance(v, (list, tuple, set))).any():
            d[c] = d[c].map(lambda v: ", ".join(map(str, v)) if isinstance(v, (list, tuple, set)) else v)
    d.columns = [str(c) for c in d.columns]
    for c in d.columns:
        if d[c].dtype == object or str(d[c].dtype).startswith("str"):
            d[c] = d[c].map(lambda v: re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", v)[:32000] if isinstance(v, str) else v)
    return d.replace([np.inf, -np.inf], np.nan)


def export_all():
    """모든 결과표를 Excel(시트별)과 UTF-8 BOM CSV로 저장하고, 기준서 파일을 따로 만듭니다."""
    sheets = {
        "읽는법": readme_sheet(),
        "1_품질점검": qc_table,
        "1_연도별수록": YEAR_DIST, "1_시군별수록": SIGUN_DIST, "1_월별수록": MONTH_VOL,
        "1_뉴스중복처리": NEWS_DUP_SUMMARY, "1_SNS관련성": REL_TABLE, "1_유효자료": VALID_SUMMARY,
        "1_인스타태그수집범위": IG_TAG_BIAS.drop(columns=[], errors="ignore"),
        "1_유튜브상투태그제거": YT_BOILER,
        **{f"3_빈도_{m}": FREQ_COMMON[m].head(1000).assign(
            영역=lambda d: d["target"].map(lambda t: AREA_MAP.get(t) or classify_area(t)[0]),
            복합명사중심=lambda d: d["target"].map(is_specific)) for m in MEDIA},
        "3_영역별다빈도": AREA_TOP,
        **{f"3_연도별_{m}": YEARLY[m] for m in MEDIA},
        "3_시군별상위": SIGUN_TOP,
        "3_보조어": AUX_TOP,
        "2_범용어판정": SPREAD_GENERIC,
        "2_정당명제외": PARTY_EXCLUDED, "2_반복생산상쇄": SRC_BOILER,
        "2_영역분류검토": AREA_REVIEW,
        "2_표현통합적용": MAPPING_TABLE, "2_표현통합제안": MERGE_SUGGEST,
        **{f"4_HOT_{m}": HOT[m] for m in MEDIA},
        **{f"4_RISING_{m}": RISING[m] for m in MEDIA},
        "4_영역별HOT": AREA_HOT, "4_영역별RISING": AREA_RISING,
        "5_반응품질": REACT_QC, "5_반응분포": REACT_DIST, "5_경과기간": AGE_EFFECT,
        "5_고반응콘텐츠": HIGH_REACT,
        **{f"5_대상반응_{p}": MVR[p] for p in SNS},
        "5_생산주체": PRODUCER_DIST,
        **{f"5_생산주체구성_{p}": PROD_MIX[p] for p in SNS},
        "6_매체간비교": CROSS, "6_영역구성비": AREA_COMP, "6_월별영역구성비": AREA_MONTHLY, "6_맥락비교": CONTEXT_CMP,
        "7_대표사례후보": REP_POOL,
        "8_최종요약표": SUMMARY,
        "4_월별추이_HOTRISING": HR_MONTHLY,
        "재현정보": ENV_INFO,
        "CONFIG": pd.DataFrame([(k, json.dumps(v, ensure_ascii=False)) for k, v in CONFIG.items()], columns=["파라미터", "값"]),
    }
    xlsx = OUT_DIR / f"미지의전북_분석결과_{RUN_TS}.xlsx"
    csv_dir = OUT_DIR / f"csv_{RUN_TS}"
    csv_dir.mkdir(exist_ok=True)
    with pd.ExcelWriter(xlsx, engine="xlsxwriter") as w:
        for name, df in sheets.items():
            d = _flatten(df)
            d.to_excel(w, sheet_name=name[:31], index=False)
            d.to_csv(csv_dir / f"{name}.csv", index=False, encoding="utf-8-sig")

    # 기준서(산출물 ②): 사전·규칙·파라미터
    spec = OUT_DIR / f"전처리_추출_분류_기준서_{RUN_TS}.xlsx"
    with pd.ExcelWriter(spec, engine="xlsxwriter") as w:
        pd.DataFrame({"지역명": sorted(REGION_WORDS)}).to_excel(w, sheet_name="지역명", index=False)
        pd.DataFrame({"불용어": sorted(STOP_WORDS)}).to_excel(w, sheet_name="불용어", index=False)
        pd.DataFrame({"보조어": sorted(AUX_WORDS)}).to_excel(w, sheet_name="보조어", index=False)
        pd.DataFrame({"화이트리스트": sorted(WHITELIST)}).to_excel(w, sheet_name="화이트리스트", index=False)
        pd.DataFrame({"정당명(제외)": sorted(PARTY_WORDS)}).to_excel(w, sheet_name="정당명_제외", index=False)
        pd.DataFrame({"SNS일반어": sorted(SNS_WORDS)}).to_excel(w, sheet_name="SNS일반어", index=False)
        MANUAL.to_excel(w, sheet_name="영역분류_수기", index=False)
        SYN.to_excel(w, sheet_name="표현통합사전", index=False)
        MAPPING_TABLE.to_excel(w, sheet_name="표현통합_적용기록", index=False)
        SPREAD_GENERIC.to_excel(w, sheet_name="범용어_자동판정", index=False)
        AREA_RULES.drop(columns="re").to_excel(w, sheet_name="영역분류규칙", index=False)
        PROD_RULES.drop(columns="re").to_excel(w, sheet_name="생산주체규칙", index=False)
        readme_sheet().to_excel(w, sheet_name="처리기준", index=False)
        sheets["CONFIG"].to_excel(w, sheet_name="파라미터", index=False)
    return xlsx, spec, csv_dir


xlsx_path, spec_path, csv_path = export_all()
print("저장 완료")
print(" -", xlsx_path)
print(" -", spec_path)
print(" -", csv_path)

## 다음 단계(검토 후 재실행)

1. `2_영역분류검토` 시트에서 `검토필요=True`와 `미분류` 대상을 대표문맥과 함께 확인하고, `dict/영역분류_수기.csv`에 확정 분류를 적습니다.
2. `2_표현통합제안`을 보고 같은 대상이면 `dict/표현통합사전.csv`에 추가합니다. 불필요한 후보는 `dict/불용어.txt`나 `dict/보조어.txt`에 추가합니다.
3. 의뢰자와 HOT·RISING 기준값(`CONFIG`)을 협의해 확정합니다.
4. **6단계부터 다시 실행**하면 바뀐 사전이 반영됩니다(형태소 분석 결과는 캐시를 그대로 씁니다). 사용자사전(Kiwi)을 바꿨다면 `cache/units_*.pkl`을 지우고 5단계부터 다시 실행하세요.